# Entrega 2 proyecto aprendizaje de máquina

Pedro Pablo Sanín Trujillo (202221527)


## 1 - Carga de datos


In [1]:
# Importar la librerías del proyecto
import pandas as pd
import tensorflow as tf
from keras.layers import TextVectorization
from sklearn.model_selection import train_test_split

Todo el proyecto vive en repositorios privados de Hugging Face, así que el notebook funciona igual en el computador local y en Colab, sin subir ni descargar archivos a mano:
 - REPO_DATOS (dataset): los datos de entrada (train.csv, eval.csv, sample_submission.csv y 50_palabras_mas_comunes/), están subidos a mano desde huggingface.co, y los submissions que genera el notebook, en la carpeta submissions/.
 - REPO_MODELOS_KERAS: los modelos Keras entrenados (<nombre>.keras y <nombre>_historial.json).
 - <usuario>/proyecto-ml-<nombre_corto>: un repositorio por cada transformer entrenado.
 Se necesita un token con permiso de escritura (https://huggingface.co/settings/tokens). Se pide con getpass la primera vez (no queda en el notebook) y después queda guardado en el computador.

In [2]:

from getpass import getpass

from huggingface_hub import HfApi, get_token, hf_hub_download, login

if get_token() is None:
    login(token=getpass("Token de Hugging Face (con permiso de escritura): "))

api = HfApi()
USUARIO_HF = api.whoami()["name"]
REPO_DATOS = f"{USUARIO_HF}/proyecto-ml-datos"
REPO_MODELOS_KERAS = f"{USUARIO_HF}/proyecto-ml-keras"
api.create_repo(REPO_MODELOS_KERAS, private=True, exist_ok=True)

print(f"Sesión de Hugging Face: {USUARIO_HF}")

Sesión de Hugging Face: pedropst05


In [ ]:
# A continuación, se cargan los conjuntos de datos de entrenamiento y evaluación

# pandas lee directamente del repositorio de Hugging Face con rutas hf://
path_train = f"hf://datasets/{REPO_DATOS}/train.csv"
path_eval = f"hf://datasets/{REPO_DATOS}/eval.csv"


dataset_train_original = pd.read_csv(path_train)
dataset_eval_original = pd.read_csv(path_eval)

dataset_train = dataset_train_original.copy()
dataset_eval = dataset_eval_original.copy()

In [ ]:
print(
    f"Número de filas y columnas del conjunto de entrenamiento: {dataset_train.shape}"
)

dataset_train.head(10)

Vemos que el dataset contiene 20466 filas, cada una con un id, la reseña correspondiente y un label que indica que si la reseña es positiva o negativa. Revisando manualmente algunas entradas, confirmamos que los únicos idiomas presentes en las reseñas son español, inglés, portugués, francés y una mezcla de español e inglés. Esto es consistente con lo indicado en el enunciado del proyecto. Preliminarmente, no separaremos los modelos reseñas por idioma pues esto reduce sustancialmente el tamaño de cada conjunto de datos. Sin embargo, si vale la pena identificar el idioma para tener una división train-test balanceada.

Además nuestra métrica de entrenamiento será el accuracy, de acuerdo con la competencia. A continuación, intentaremos identificar el idioma de cada una de nuestras reseñas.


## 2 - Identificando el idioma de la reseña

Para esto, descargaremos las 50 palabras más comunes en cada uno de los idiomas. Por la ley de Zipf, si las listas de las 50 palabras más comunes son distintas, una de las palabras aparecerá en la lista y esto nos servirá para determinar el idioma de la reseña. Asimismo, las palabras más comunes en cada idioma son artículos y generalmente tienen el mismo significado.

Así, el criterio para determinar el idioma de una reseña será el siguiente:

1. El primer y segundo idioma candidato serán los primeros dos conjuntos de palabras que tengan más coincidencia con las palabras de la reseña.

2. Con las palabras que pertenecen al conjunto más común,


In [ ]:
import sys
import unicodedata
from collections import Counter

# Tabla de traducción que reemplaza por un espacio todo carácter Unicode de puntuación
# (categorías 'P*'), lo que incluye ¿ ¡ « » “ ” … además de la puntuación ASCII.
puntua = {
    i: " "
    for i in range(sys.maxunicode + 1)
    if unicodedata.category(chr(i)).startswith("P")
}

# Palabras de la lista de inglés que también son muy comunes en español ("lo he usado"),
# por lo que no cuentan como evidencia de inglés dentro de una reseña en español.
AMBIGUAS_EN = {"he"}


def procesar_resena(text):
    """Pasa la reseña a minúsculas, elimina la puntuación y la separa en palabras."""
    text = text.lower()
    text = text.translate(puntua)
    return text.split()


def palabras_exclusivas(palabras_por_idioma):
    """Para cada idioma, las palabras comunes que no aparecen en la lista de ningún otro idioma."""
    return {
        idioma: palabras
        - set().union(
            *(otras for otro, otras in palabras_por_idioma.items() if otro != idioma)
        )
        for idioma, palabras in palabras_por_idioma.items()
    }


def identificar_idioma(resena, palabras_por_idioma, exclusivas):
    """Identifica el idioma de una reseña: 'es', 'en', 'fr', 'pt_br', 'es_en' (bilingüe) o None.

    1. El idioma principal es el que tiene más ocurrencias de sus palabras exclusivas. Si la reseña
       no tiene ninguna palabra exclusiva, se usan todas las palabras comunes (incluidas las compartidas).
    2. Si el idioma principal es español o inglés y la reseña contiene al menos una palabra exclusiva
       del otro idioma, la reseña se considera bilingüe ('es_en').
    """
    conteo = Counter(procesar_resena(resena))

    def puntajes(listas):
        return {
            idioma: sum(n for palabra, n in conteo.items() if palabra in palabras)
            for idioma, palabras in listas.items()
        }

    puntaje = puntajes(exclusivas)
    if max(puntaje.values()) == 0:
        puntaje = puntajes(palabras_por_idioma)
        if max(puntaje.values()) == 0:
            return None
    principal = max(puntaje, key=lambda k: puntaje[k])

    hay_es = any(palabra in exclusivas["es"] for palabra in conteo)
    hay_en = any(palabra in exclusivas["en"] - AMBIGUAS_EN for palabra in conteo)
    if principal in ("es", "en") and hay_es and hay_en:
        return "es_en"
    return principal

In [ ]:
# Cargamos las 50 palabras más comunes por idioma y les aplicamos la misma limpieza que a las reseñas,
# para que entradas como "c'", "l'" o "'s" queden como "c", "l" y "s" y puedan coincidir con las reseñas.
path_palabras = f"hf://datasets/{REPO_DATOS}/50_palabras_mas_comunes/50_palabras_mas_comunes.csv"

palabras_comunes = pd.read_csv(path_palabras)
palabras_comunes["palabra"] = palabras_comunes["palabra"].map(
    lambda p: procesar_resena(p)[0]
)

# Conjunto de palabras por idioma, y el subconjunto de palabras exclusivas de cada idioma
palabras_por_idioma = palabras_comunes.groupby("idioma")["palabra"].apply(set).to_dict()
exclusivas = palabras_exclusivas(palabras_por_idioma)

palabras_comunes.head()

In [ ]:
print(
    identificar_idioma("Esta es una reseña de prueba", palabras_por_idioma, exclusivas)
)
print(identificar_idioma("This is a test review", palabras_por_idioma, exclusivas))
print(
    identificar_idioma("Ceci est une critique de test", palabras_por_idioma, exclusivas)
)
print(
    identificar_idioma("Esta é uma crítica de teste", palabras_por_idioma, exclusivas)
)
print(
    identificar_idioma(
        "Esta es una reseña de prueba en inglés: This is a test review",
        palabras_por_idioma,
        exclusivas,
    )
)
print(
    identificar_idioma(
        "Esta es una reseña de prueba en francés: Ceci est une critique de test",
        palabras_por_idioma,
        exclusivas,
    )
)

In [ ]:
# Identificamos el idioma de cada reseña de entrenamiento y evaluación
dataset_train["idioma"] = dataset_train["text"].map(
    lambda t: identificar_idioma(t, palabras_por_idioma, exclusivas)
)
dataset_eval["idioma"] = dataset_eval["text"].map(
    lambda t: identificar_idioma(t, palabras_por_idioma, exclusivas)
)

dataset_train["idioma"].value_counts(dropna=False, normalize=True)

In [ ]:
dataset_train.head()

Ahora, dividimos el conjunto de datos en prueba y entrenamiento estratificado para incluir una proporción representativa de cada idioma y de cada etiqueta. Para esto, usaremos `sklearn.train_test_split`.


In [ ]:
# Codificamos las etiquetas de como enteros
etiquetas = sorted(
    dataset_train["label"].unique()
)  # ['negativo', 'neutral', 'positivo']
etiqueta_a_id = {e: i for i, e in enumerate(etiquetas)}


# División 90/10 estratificada por idioma y etiqueta a la vez: cada combinación
# (p. ej. "fr|neutral") conserva la misma proporción en entrenamiento y prueba.
estrato = dataset_train["idioma"].astype(str) + "|" + dataset_train["label"]

# train_test_split exige al menos 2 reseñas por estrato. Las combinaciones con una sola
# reseña (solo hay una reseña bilingüe es_en neutral) no se pueden dividir, así que van
# directamente a entrenamiento.
raros = estrato.map(estrato.value_counts()) < 2

train_df, test_df = train_test_split(
    dataset_train[~raros],
    test_size=0.1,
    random_state=42,
    stratify=estrato[~raros],
)
train_df = pd.concat([train_df, dataset_train[raros]])


def dataset_a_tf(df, shuffle):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["text"].values, df["label"].map(etiqueta_a_id).values)
    )
    if shuffle:
        ds = ds.shuffle(len(df), seed=42)
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)


train_dataset = dataset_a_tf(train_df, shuffle=True)
test_dataset = dataset_a_tf(test_df, shuffle=False)

## 2 - Red neuronal


In [ ]:
encoder_mlp = TextVectorization(output_mode="tf_idf")
encoder_mlp.adapt(train_df["text"])

In [ ]:
model_mlp = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(), dtype=tf.string),
        encoder_mlp,
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(3),
    ]
)

model_mlp.summary()

In [ ]:
model_mlp.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

Ahora para entrenar el modelo, usaremos `tf.keras.callbacks.EarlyStopping`. Esto nos permite parar el entrenamiento cuando la pérdida de validación (`val_loss`) deja de disminuir, y restaurar los parámetros de la época con menor pérdida de validación. Usamos la pérdida y no el accuracy porque el accuracy de validación puede seguir subiendo levemente mientras el modelo ya se está sobreajustando (la pérdida sube porque el modelo se vuelve demasiado confiado en sus errores).


In [ ]:
# En lugar de fijar el número de épocas, entrenamos hasta un máximo de 50 y detenemos el
# entrenamiento cuando la métrica `monitor` de validación deja de mejorar durante `patience`
# épocas seguidas. restore_best_weights=True deja el modelo con los pesos de la mejor época.
# Por defecto se monitorea la pérdida de validación ("val_loss"), que se minimiza; si se pasa
# un accuracy (p. ej. monitor="val_accuracy"), se maximiza.
def entrenar_modelo_con_parada_temprana(
    model, train_dataset, test_dataset, max_epochs=50, patience=3, monitor="val_loss"
):
    modo = "min" if "loss" in monitor else "max"
    parada_temprana = tf.keras.callbacks.EarlyStopping(
        monitor=monitor,
        mode=modo,
        patience=patience,
        restore_best_weights=True,
        verbose=1,  # informa en qué época se detuvo y de qué época restauró los pesos
    )

    history = model.fit(
        train_dataset,
        epochs=max_epochs,
        validation_data=test_dataset,
        callbacks=[parada_temprana],
    )

    # Guardamos en el historial la época cuyos pesos restauró la parada temprana (índice desde 0),
    # para que los resúmenes usen exactamente esa época y no la recalculen con otra métrica.
    history.mejor_epoca = parada_temprana.best_epoch
    print(
        f"Mejor época según {monitor}: {history.mejor_epoca + 1} "
        f"(val_loss = {history.history['val_loss'][history.mejor_epoca]:.4f}, "
        f"val_accuracy = {history.history['val_accuracy'][history.mejor_epoca]:.4f})"
    )

    return history


# ---------------------------------------------------------------------------------------------
# Guardado y carga de modelos
# ---------------------------------------------------------------------------------------------
# Cada modelo entrenado se sube a REPO_MODELOS_KERAS como <nombre>.keras (model.save), junto con su
# historial en <nombre>_historial.json. Al volver a ejecutar el notebook, si el modelo ya está en el
# Hub se descarga y se carga con keras.models.load_model en lugar de entrenarlo de nuevo.
# Con REENTRENAR = True se ignoran los modelos guardados y se vuelve a entrenar todo.
import json
import os
import tempfile
from types import SimpleNamespace

from huggingface_hub import CommitOperationAdd

REENTRENAR = False


def historial_a_json(history):
    """Métricas por época y época restaurada por la parada temprana, como bytes JSON."""
    datos = {
        "history": {k: [float(v) for v in vals] for k, vals in history.history.items()},
        "mejor_epoca": int(history.mejor_epoca),
    }
    return json.dumps(datos, indent=2).encode()


def descargar_historial(repo, archivo):
    """Devuelve un objeto con `.history` y `.mejor_epoca`, igual que el que devuelve el entrenamiento."""
    with open(hf_hub_download(repo, archivo)) as f:
        datos = json.load(f)
    return SimpleNamespace(history=datos["history"], mejor_epoca=datos["mejor_epoca"])


def resumen_mejor_epoca(history):
    return (
        f"mejor época: {history.mejor_epoca + 1}, "
        f"val_loss = {history.history['val_loss'][history.mejor_epoca]:.4f}, "
        f"val_accuracy = {history.history['val_accuracy'][history.mejor_epoca]:.4f}"
    )


def entrenar_o_cargar(nombre, construir_modelo, train_dataset, test_dataset, **kwargs):
    """Carga el modelo `nombre` de REPO_MODELOS_KERAS si ya está ahí; si no, lo construye con
    `construir_modelo()`, lo entrena con parada temprana (`kwargs` se pasan al entrenamiento) y lo
    sube. Devuelve (modelo, historial)."""
    archivo_modelo = f"{nombre}.keras"
    archivo_historial = f"{nombre}_historial.json"

    if not REENTRENAR and api.file_exists(REPO_MODELOS_KERAS, archivo_historial):
        ruta_modelo = hf_hub_download(REPO_MODELOS_KERAS, archivo_modelo)
        try:
            model = tf.keras.models.load_model(ruta_modelo)
        except Exception as e:
            # Un .keras guardado con una versión de Keras más nueva (p. ej. local) puede traer
            # opciones de configuración que una versión más antigua (p. ej. Colab) no reconoce.
            # Los pesos sí son compatibles: se reconstruye la arquitectura con el código del
            # notebook y se cargan solo los pesos del archivo.
            print(f"load_model falló ({type(e).__name__}); se reconstruye el modelo y se cargan sus pesos")
            model = construir_modelo()
            model.load_weights(ruta_modelo)
        history = descargar_historial(REPO_MODELOS_KERAS, archivo_historial)
        print(f"{nombre} cargado de Hugging Face ({resumen_mejor_epoca(history)})")
        return model, history

    model = construir_modelo()
    history = entrenar_modelo_con_parada_temprana(model, train_dataset, test_dataset, **kwargs)

    with tempfile.TemporaryDirectory() as carpeta:
        ruta_modelo = os.path.join(carpeta, archivo_modelo)
        model.save(ruta_modelo)
        # Un solo commit con los dos archivos; el historial es el que marca que el modelo está completo
        api.create_commit(
            REPO_MODELOS_KERAS,
            operations=[
                CommitOperationAdd(archivo_modelo, ruta_modelo),
                CommitOperationAdd(archivo_historial, historial_a_json(history)),
            ],
            commit_message=f"Modelo {nombre}",
        )
    print(f"{nombre} subido a https://huggingface.co/{REPO_MODELOS_KERAS}")
    return model, history

In [ ]:
model_mlp, historial_mlp = entrenar_o_cargar(
    "mlp_tfidf", lambda: model_mlp, train_dataset, test_dataset
)

## 3 - Red neuronal recurrente básica


Inicialmente, usaremos una red neuronal recurrente para clasificar las reseñas. Inicialmente, veremos el desempeño de una RNN simple, donde cada neurona tiene información sobre las operaciones que ha hecho en pasos anteriores. Las redes neuronales recurrentes funcionan bien al procesar lenguaje natural pues el lenguaje tiene una naturaleza secuencial.

Usaremos la implementación de RNNs dada por `tf.keras.layers.SimpleRNN`. Para que el texto pueda pasar exitosamente por una red neuronal, debemos vectorizarlo. Para esto, usaremos una capa inicial de la red neuronal dada por `tf.keras.layers.TextVectorization`. Este algoritmo de vectorización combina varias formas de vectorizar frases como _tf_idf_, _count_ (bag of words) y soporta ngramas.


In [ ]:
# Para una red recurrente necesitamos una secuencia de índices de palabras (output_mode='int'),
# no un vector tf_idf. Cada reseña se rellena o recorta a 100 palabras (mediana: 57, máximo: 125).


# Tildes a quitar (unicodedata no sirve aquí porque Keras pasa tensores, no str,
# y TensorFlow no trae normalización Unicode sin el paquete tensorflow_text).
TILDES = {
    "[áàâäã]": "a",
    "[éèêë]": "e",
    "[íìîï]": "i",
    "[óòôöõ]": "o",
    "[úùûü]": "u",
    "ñ": "n",
    "ç": "c",
}


# Registrar la función permite guardar el TextVectorization en .keras y volver a cargarlo.
@tf.keras.utils.register_keras_serializable(package="proyecto")
def standardize_text(text):
    text = tf.strings.lower(text, encoding="utf-8")
    text = tf.strings.regex_replace(text, "<br />", " ")
    for patron, letra in TILDES.items():
        text = tf.strings.regex_replace(text, patron, letra)
    text = tf.strings.regex_replace(text, "[^a-z']", " ")
    return text


def crear_encoder_rnn(train_df, n_gram=1):

    encoder_rnn = TextVectorization(
        output_mode="int",
        output_sequence_length=100,
        standardize=standardize_text,
        ngrams=n_gram,  # pyright: ignore[reportArgumentType]
    )
    encoder_rnn.adapt(train_df["text"])
    return encoder_rnn


encoder_rnn = crear_encoder_rnn(train_df)

texto_original = train_df["text"].iloc[0]
texto_vectorizado = encoder_rnn(tf.convert_to_tensor([texto_original]))

encoder_rnn = crear_encoder_rnn(train_df)

texto_original = train_df["text"][1]
texto_vectorizado = encoder_rnn(tf.convert_to_tensor([texto_original]))
print(f"Texto original: {texto_original}")
print(f"Texto vectorizado: {texto_vectorizado}")

Ahora, entrenaremos una red neuronal con una capa de embedding, una capa SimpleRNN de 64 neuronas y una capa oculta densa de 32 neuronas con función de activación ReLU. Asimismo, entrenamos un embedding con una dimensión de salida de 64.


In [ ]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(), dtype=tf.string),
        encoder_rnn,
        tf.keras.layers.Embedding(
            input_dim=encoder_rnn.vocabulary_size(), output_dim=64, mask_zero=True
        ),
        tf.keras.layers.SimpleRNN(64),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(3),
    ]
)

model.summary()

In [ ]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

In [ ]:
model, historial_simplernn = entrenar_o_cargar(
    "simplernn", lambda: model, train_dataset, test_dataset
)

### 3.1 - Probando diferentes combinaciones de capas

A continuación, probaremos 5 combinaciones distintas de capas para determinar cual tiene mejor desempeño en validación.


Para no repetir código, definimos una función que arma la red con la misma estructura de la celda anterior
(entrada de texto → `encoder_rnn` → `Embedding` → **capas recurrentes** → `Dense(32, relu)` → `Dense(3)`)
y solo cambia el bloque de capas recurrentes. Cada modelo tiene su propia capa de embedding, de modo que
ninguno comparte pesos entrenables con otro; `encoder_rnn` sí se comparte, pero no tiene pesos entrenables.


In [ ]:
def construir_modelo_rnn(capas_recurrentes, nombre):
    """Arma y compila una red recurrente cuyo bloque central son las capas recibidas."""
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(), dtype=tf.string),
            encoder_rnn,
            tf.keras.layers.Embedding(
                input_dim=encoder_rnn.vocabulary_size(), output_dim=64, mask_zero=True
            ),
            *capas_recurrentes,
            tf.keras.layers.Dense(32, activation="relu"),
            tf.keras.layers.Dense(3),
        ],
        name=nombre,
    )
    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.Adam(),
        metrics=["accuracy"],
    )
    return model


# Guardamos el historial de entrenamiento de cada combinación para compararlas al final
historiales = {}

#### Modelo 1 - SimpleRNN bidireccional

La misma `SimpleRNN(64)` del modelo base, pero envuelta en `Bidirectional`: una copia lee la reseña de
izquierda a derecha y otra de derecha a izquierda, y sus salidas se concatenan. Así, la representación final
no depende solo de las últimas palabras de la reseña.


In [ ]:
model_rnn_1, historiales["1 - SimpleRNN bidireccional"] = entrenar_o_cargar(
    "simplernn_bidireccional",
    lambda: construir_modelo_rnn(
        [tf.keras.layers.Bidirectional(tf.keras.layers.SimpleRNN(64))],
        nombre="simplernn_bidireccional",
    ),
    train_dataset,
    test_dataset,
)
model_rnn_1.summary()

#### Modelo 2 - Dos SimpleRNN apiladas

Dos capas `SimpleRNN` apiladas (64 y 32 neuronas). La primera usa `return_sequences=True` para entregarle a la
segunda la salida de cada paso de tiempo, no solo la del último.


In [ ]:
model_rnn_2, historiales["2 - SimpleRNN apiladas"] = entrenar_o_cargar(
    "simplernn_apiladas",
    lambda: construir_modelo_rnn(
        [
            tf.keras.layers.SimpleRNN(64, return_sequences=True),
            tf.keras.layers.SimpleRNN(32),
        ],
        nombre="simplernn_apiladas",
    ),
    train_dataset,
    test_dataset,
)
model_rnn_2.summary()

#### Modelo 3 - LSTM

Reemplazamos la `SimpleRNN` por una `LSTM(64)`. Sus compuertas de entrada, olvido y salida le permiten conservar
información a lo largo de secuencias largas y mitigan el desvanecimiento del gradiente de la RNN simple.


In [ ]:
model_rnn_3, historiales["3 - LSTM"] = entrenar_o_cargar(
    "lstm",
    lambda: construir_modelo_rnn(
        [tf.keras.layers.LSTM(64)],
        nombre="lstm",
    ),
    train_dataset,
    test_dataset,
)
model_rnn_3.summary()

#### Modelo 4 - GRU bidireccional con dropout

Una `GRU(64)` bidireccional (similar a la LSTM pero con menos parámetros) seguida de un `Dropout(0.5)` para
regularizar, ya que con alrededor de 18.000 reseñas de entrenamiento los modelos se sobreajustan con facilidad.


In [ ]:
model_rnn_4, historiales["4 - GRU bidireccional + dropout"] = entrenar_o_cargar(
    "gru_bidireccional_dropout",
    lambda: construir_modelo_rnn(
        [
            tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64)),
            tf.keras.layers.Dropout(0.5),
        ],
        nombre="gru_bidireccional_dropout",
    ),
    train_dataset,
    test_dataset,
)
model_rnn_4.summary()

#### Modelo 5 - Dos LSTM bidireccionales apiladas con dropout

La combinación más profunda: dos `LSTM` bidireccionales apiladas (64 y 32 neuronas), con `Dropout(0.3)` entre ellas
y `Dropout(0.5)` antes de las capas densas.


In [ ]:
model_rnn_5, historiales["5 - LSTM bidireccionales apiladas"] = entrenar_o_cargar(
    "lstm_bidireccionales_apiladas",
    lambda: construir_modelo_rnn(
        [
            tf.keras.layers.Bidirectional(
                tf.keras.layers.LSTM(64, return_sequences=True)
            ),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
            tf.keras.layers.Dropout(0.5),
        ],
        nombre="lstm_bidireccionales_apiladas",
    ),
    train_dataset,
    test_dataset,
)
model_rnn_5.summary()

#### Creación de archivos submission


In [ ]:
# Métricas de la época que restauró la parada temprana (la de menor val_loss)
resumen_rnn = pd.DataFrame(
    [
        {
            "modelo": nombre,
            "mejor_epoca": h.mejor_epoca + 1,
            "val_loss": h.history["val_loss"][h.mejor_epoca],
            "val_accuracy": h.history["val_accuracy"][h.mejor_epoca],
            "train_accuracy": h.history["accuracy"][h.mejor_epoca],
        }
        for nombre, h in historiales.items()
    ]
).sort_values("val_accuracy", ascending=False)

resumen_rnn

In [ ]:
# Si un submission ya está en REPO_DATOS (submissions/<archivo>), no se vuelve a crear: se descarga
# el que ya existe y, en crear_archivo_predicciones, ni siquiera se predice con el modelo. Con
# REGENERAR_SUBMISSIONS = True, o con REENTRENAR = True (los modelos cambian), se vuelven a crear
# y se sobrescriben.
REGENERAR_SUBMISSIONS = False


def submission_existente(archivo_salida):
    """El submission `archivo_salida` de REPO_DATOS si ya existe y no hay que regenerarlo; si no, None."""
    ruta = f"submissions/{archivo_salida}"
    if REGENERAR_SUBMISSIONS or REENTRENAR:
        return None
    if not api.file_exists(REPO_DATOS, ruta, repo_type="dataset"):
        return None
    print(f"{archivo_salida} ya existe en Hugging Face; no se vuelve a crear")
    return pd.read_csv(f"hf://datasets/{REPO_DATOS}/{ruta}")


def _subir_submission(clases, df, archivo_salida):
    """Sube a REPO_DATOS, en submissions/<archivo_salida>, un CSV con el formato de
    sample_submission.csv: columnas `id` y `answer` ('negativo', 'neutral' o 'positivo').
    `clases` son las clases predichas (0, 1 o 2) para cada reseña de `df`."""
    submission = pd.DataFrame(
        {"id": df["id"].values, "answer": [etiquetas[c] for c in clases]}
    )
    api.upload_file(
        path_or_fileobj=submission.to_csv(index=False).encode(),
        path_in_repo=f"submissions/{archivo_salida}",
        repo_id=REPO_DATOS,
        repo_type="dataset",
        commit_message=f"Submission {archivo_salida}",
    )

    print(
        f"{archivo_salida} subido a https://huggingface.co/datasets/{REPO_DATOS}/tree/main/submissions "
        f"({len(submission)} predicciones)"
    )
    print(submission["answer"].value_counts(normalize=True))
    return submission


def subir_submission(clases, df, archivo_salida):
    """Sube el submission con las clases ya predichas, salvo que ya exista en Hugging Face."""
    existente = submission_existente(archivo_salida)
    if existente is not None:
        return existente
    return _subir_submission(clases, df, archivo_salida)


def crear_archivo_predicciones(
    model, df, archivo_salida="submission.csv", columnas=("text",)
):
    """Predice la etiqueta de cada reseña de `df` y sube el submission, salvo que ya exista en
    Hugging Face (en ese caso no se predice y se devuelve el existente).

    `columnas` son las columnas de `df` que recibe el modelo: con una sola (por defecto "text")
    se le pasan los textos directamente; con varias, un dict {nombre_columna: textos}, que es lo
    que esperan los modelos de dos entradas de la sección 3.2."""
    existente = submission_existente(archivo_salida)
    if existente is not None:
        return existente

    if len(columnas) == 1:
        entradas = df[columnas[0]].values
    else:
        entradas = {c: df[c].values for c in columnas}
    textos = tf.data.Dataset.from_tensor_slices(entradas).batch(32)

    # El modelo devuelve logits (una fila de 3 valores por reseña): la clase predicha es la de
    # mayor logit, y la traducimos de vuelta a texto con la misma lista usada al codificarlas.
    logits = model.predict(textos)
    return _subir_submission(logits.argmax(axis=1), df, archivo_salida)

In [ ]:
# Cambiar por el modelo con mejor val_accuracy
modelos_rnn = [model_rnn_1, model_rnn_2, model_rnn_3, model_rnn_4, model_rnn_5]
nombres_modelo = ["Simple RNN", "Simple RNNs apiladas", "LSTM", "GRU", "LSTM apiladas"]
modelos_rnn = {"nombre": nombres_modelo, "modelo": modelos_rnn}

for nombre, modelo in zip(modelos_rnn["nombre"], modelos_rnn["modelo"]):
    submission = crear_archivo_predicciones(
        modelo, dataset_eval, f"submission_{nombre}.csv"
    )
    submission.head()

### 3.2 - Ajustando y regularizando los modelos RNN

En el proyecto anterior, la última oración era la que más significado daba a la reseña completa. Para esto, agregabamos la última oración como una entrada adicional al modelo. En este caso, haremos lo mismo.

Asimismo, vemos que los modelos que estamos usando tienden al overfitting. Para solucionar esto intentaremos pararemos el entrenamiento en las epochs en función de la función de pérdida y no el accuracy en el entrenamiento. De esta forma, el modelo elegido no se elige de acuerdo a los datos específicos de entrenamiento.

También, intentaremos regularizar más las RNNs. Para esto, usaremos `SpatialDropout1D` después del embedding. A diferencia de `Dropout`, `SpatialDropout1D` hace dropout para columnas enteras del espacio, lo que aumenta la regularización.


In [ ]:
import re


def ultima_oracion(texto):
    """Devuelve la última oración de la reseña (corta después de . ! ?)."""
    oraciones = [o for o in re.split(r"(?<=[.!?])\s+", texto.strip()) if o]
    return oraciones[-1] if oraciones else texto


# .assign crea un DataFrame nuevo y evita el SettingWithCopyWarning
train_df = train_df.assign(ultima=train_df["text"].map(ultima_oracion))
test_df = test_df.assign(ultima=test_df["text"].map(ultima_oracion))
dataset_eval = dataset_eval.assign(ultima=dataset_eval["text"].map(ultima_oracion))


def entradas_modelo(df, columnas):
    """Una columna -> arreglo de textos (como antes); varias -> dict {nombre_columna: textos}."""
    if len(columnas) == 1:
        return df[columnas[0]].values
    return {c: df[c].values for c in columnas}


def dataset_a_tf(df, shuffle, columnas=("text",)):
    ds = tf.data.Dataset.from_tensor_slices(
        (entradas_modelo(df, columnas), df["label"].map(etiqueta_a_id).values)
    )
    if shuffle:
        ds = ds.shuffle(len(df), seed=42)
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)


train_dataset = dataset_a_tf(train_df, shuffle=True)
test_dataset = dataset_a_tf(test_df, shuffle=False)

Para comparar de forma justa, cada modelo mejorado conserva exactamente el bloque recurrente de su
modelo original de la sección 2.1. Solo cambian dos cosas:

1. Una segunda entrada con la última oración de la reseña, procesada por su propia copia del bloque recurrente.
2. `SpatialDropout1D(0.3)` después del embedding.

La parada temprana según `val_loss` ya no es una diferencia entre ambos grupos: `entrenar_modelo_con_parada_temprana`
la usa por defecto, así que los modelos originales también se eligen con la pérdida de validación.


In [ ]:
def construir_modelo_rnn_mejorado(
    crear_capas_recurrentes, nombre, tasa_spatial_dropout=0.3
):
    """Igual que `construir_modelo_rnn`, pero con las mejoras de esta sección:

    - Dos entradas: la reseña completa (`text`) y su última oración (`ultima`). Cada una pasa
      por su propia copia del bloque recurrente y las dos salidas se concatenan antes de las
      capas densas.
    - `SpatialDropout1D` justo después del embedding, en ambas ramas.

    `crear_capas_recurrentes` es una función que devuelve una lista NUEVA de capas cada vez que se
    llama, para que cada rama tenga sus propios pesos (si se usaran las mismas capas en las dos
    ramas, Keras compartiría sus pesos). El embedding sí se comparte entre ambas ramas, de modo
    que las palabras de la última oración también aprenden de la reseña completa.
    """
    embedding = tf.keras.layers.Embedding(
        input_dim=encoder_rnn.vocabulary_size(), output_dim=64, mask_zero=True
    )

    def rama(entrada):
        x = embedding(encoder_rnn(entrada))
        x = tf.keras.layers.SpatialDropout1D(tasa_spatial_dropout)(x)
        for capa in crear_capas_recurrentes():
            x = capa(x)
        return x

    # Los nombres de los Input coinciden con las columnas del DataFrame (las claves del dict)
    texto = tf.keras.layers.Input(shape=(), dtype=tf.string, name="text")
    ultima = tf.keras.layers.Input(shape=(), dtype=tf.string, name="ultima")

    x = tf.keras.layers.Concatenate()([rama(texto), rama(ultima)])
    x = tf.keras.layers.Dense(32, activation="relu")(x)
    salida = tf.keras.layers.Dense(3)(x)

    model = tf.keras.Model([texto, ultima], salida, name=nombre)
    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.Adam(),
        metrics=["accuracy"],
    )
    return model


# Conjuntos de datos con las dos entradas: reseña completa y última oración
columnas_2 = ("text", "ultima")
train_dataset_2 = dataset_a_tf(train_df, shuffle=True, columnas=columnas_2)
test_dataset_2 = dataset_a_tf(test_df, shuffle=False, columnas=columnas_2)

# Usamos las mismas claves que en `historiales` para poder comparar cada modelo con su original
historiales_mejorados = {}

#### Modelo 1 mejorado - SimpleRNN bidireccional

Mismo bloque recurrente que el modelo 1: una `SimpleRNN(64)` bidireccional.


In [ ]:
model_rnn_1_mejorado, historiales_mejorados["1 - SimpleRNN bidireccional"] = (
    entrenar_o_cargar(
        "simplernn_bidireccional_mejorado",
        lambda: construir_modelo_rnn_mejorado(
            lambda: [tf.keras.layers.Bidirectional(tf.keras.layers.SimpleRNN(64))],
            nombre="simplernn_bidireccional_mejorado",
        ),
        train_dataset_2,
        test_dataset_2,
        monitor="val_loss",
    )
)
model_rnn_1_mejorado.summary()

#### Modelo 2 mejorado - Dos SimpleRNN apiladas

Mismo bloque recurrente que el modelo 2: `SimpleRNN(64, return_sequences=True)` seguida de `SimpleRNN(32)`.


In [ ]:
model_rnn_2_mejorado, historiales_mejorados["2 - SimpleRNN apiladas"] = (
    entrenar_o_cargar(
        "simplernn_apiladas_mejorado",
        lambda: construir_modelo_rnn_mejorado(
            lambda: [
                tf.keras.layers.SimpleRNN(64, return_sequences=True),
                tf.keras.layers.SimpleRNN(32),
            ],
            nombre="simplernn_apiladas_mejorado",
        ),
        train_dataset_2,
        test_dataset_2,
        monitor="val_loss",
    )
)
model_rnn_2_mejorado.summary()

#### Modelo 3 mejorado - LSTM

Mismo bloque recurrente que el modelo 3: una `LSTM(64)`.


In [ ]:
model_rnn_3_mejorado, historiales_mejorados["3 - LSTM"] = entrenar_o_cargar(
    "lstm_mejorado",
    lambda: construir_modelo_rnn_mejorado(
        lambda: [tf.keras.layers.LSTM(64)],
        nombre="lstm_mejorado",
    ),
    train_dataset_2,
    test_dataset_2,
    monitor="val_loss",
)
model_rnn_3_mejorado.summary()

#### Modelo 4 mejorado - GRU bidireccional con dropout

Mismo bloque recurrente que el modelo 4: una `GRU(64)` bidireccional seguida de `Dropout(0.5)`.


In [ ]:
model_rnn_4_mejorado, historiales_mejorados["4 - GRU bidireccional + dropout"] = (
    entrenar_o_cargar(
        "gru_bidireccional_dropout_mejorado",
        lambda: construir_modelo_rnn_mejorado(
            lambda: [
                tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64)),
                tf.keras.layers.Dropout(0.5),
            ],
            nombre="gru_bidireccional_dropout_mejorado",
        ),
        train_dataset_2,
        test_dataset_2,
        monitor="val_loss",
    )
)
model_rnn_4_mejorado.summary()

#### Modelo 5 mejorado - Dos LSTM bidireccionales apiladas con dropout

Mismo bloque recurrente que el modelo 5: dos `LSTM` bidireccionales apiladas (64 y 32 neuronas) con `Dropout(0.3)` entre ellas y `Dropout(0.5)` al final.


In [ ]:
model_rnn_5_mejorado, historiales_mejorados["5 - LSTM bidireccionales apiladas"] = (
    entrenar_o_cargar(
        "lstm_bidireccionales_apiladas_mejorado",
        lambda: construir_modelo_rnn_mejorado(
            lambda: [
                tf.keras.layers.Bidirectional(
                    tf.keras.layers.LSTM(64, return_sequences=True)
                ),
                tf.keras.layers.Dropout(0.3),
                tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
                tf.keras.layers.Dropout(0.5),
            ],
            nombre="lstm_bidireccionales_apiladas_mejorado",
        ),
        train_dataset_2,
        test_dataset_2,
        monitor="val_loss",
    )
)
model_rnn_5_mejorado.summary()

#### Comparación con los modelos originales

Para cada modelo tomamos las métricas de la época que restauró la parada temprana. Tanto en los originales como
en los mejorados esa época se eligió con `val_loss`, así que el `val_accuracy` reportado no está sesgado por la
selección y ambos grupos se comparan en igualdad de condiciones. Con ~2.000 reseñas de validación, el error estándar
del accuracy es de casi 1 punto, así que diferencias menores a eso no son concluyentes.


In [ ]:
def resumir_historiales(historiales):
    """Una fila por modelo con las métricas de la época restaurada por la parada temprana."""
    filas = []
    for nombre, h in historiales.items():
        mejor = h.mejor_epoca
        filas.append(
            {
                "modelo": nombre,
                "mejor_epoca": mejor + 1,
                "val_accuracy": h.history["val_accuracy"][mejor],
                "val_loss": h.history["val_loss"][mejor],
                "train_accuracy": h.history["accuracy"][mejor],
            }
        )
    return pd.DataFrame(filas)


comparacion = resumir_historiales(historiales).merge(
    resumir_historiales(historiales_mejorados),
    on="modelo",
    suffixes=("_original", "_mejorado"),
)
comparacion["diferencia_val_accuracy"] = (
    comparacion["val_accuracy_mejorado"] - comparacion["val_accuracy_original"]
)

comparacion[
    [
        "modelo",
        "mejor_epoca_original",
        "mejor_epoca_mejorado",
        "val_loss_original",
        "val_loss_mejorado",
        "val_accuracy_original",
        "val_accuracy_mejorado",
        "diferencia_val_accuracy",
        "train_accuracy_original",
        "train_accuracy_mejorado",
    ]
].sort_values(by=["val_accuracy_mejorado"], ascending=[False])

In [ ]:
# Archivos submission de los modelos mejorados (reciben la reseña y su última oración)
modelos_mejorados = {
    "Simple RNN": model_rnn_1_mejorado,
    "Simple RNNs apiladas": model_rnn_2_mejorado,
    "LSTM": model_rnn_3_mejorado,
    "GRU": model_rnn_4_mejorado,
    "LSTM apiladas": model_rnn_5_mejorado,
}

for nombre, modelo in modelos_mejorados.items():
    crear_archivo_predicciones(
        modelo,
        dataset_eval,
        f"submission_{nombre}_mejorado.csv",
        columnas=columnas_2,
    )

Vemos entonces que estas estrategias de regularización no funcionaron.


### 3.3 - Identificación del error

Para entender en qué se equivocan las RNN, analizamos sobre `test_df` el modelo con mejor `val_accuracy` de las
secciones 3.1 y 3.2:

1. Precisión, recall y F1 de cada clase.
2. La matriz de confusión normalizada por fila: qué fracción de las reseñas de cada clase real termina en cada
   clase predicha.
3. El accuracy por idioma, usando la columna `idioma` de la sección 2.
4. Una muestra de reseñas mal clasificadas, para leerlas completas.

`identificar_errores` solo recibe las clases predichas, así que la reutilizamos tal cual con mBERT en la sección 4.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, classification_report


def graficar_perdida(historial, nombre):
    """Pérdida de entrenamiento y de validación por época, marcando la época que restauró la
    parada temprana."""
    epocas = range(1, len(historial.history["val_loss"]) + 1)
    plt.plot(epocas, historial.history["loss"], marker="o", label="loss (entrenamiento)")
    plt.plot(epocas, historial.history["val_loss"], marker="o", label="val_loss (validación)")
    plt.axvline(
        historial.mejor_epoca + 1, color="gray", linestyle="--",
        label=f"mejor época ({historial.mejor_epoca + 1})",
    )
    plt.xticks(list(epocas))
    plt.title(f"Función de pérdida por época - {nombre}")
    plt.xlabel("Época")
    plt.ylabel("Entropía cruzada")
    plt.legend()
    plt.show()


def evaluar_modelo_validacion(
    clases, df, nombre, n_ejemplos=10, repo_historial=None, archivo_historial="historial.json"
):
    """Analiza las predicciones `clases` (0, 1 o 2) sobre `df` y devuelve las reseñas mal clasificadas.

    Si se pasa `repo_historial`, descarga de ese repositorio de Hugging Face el historial
    `archivo_historial` y grafica loss y val_loss por época. Los transformers guardan su historial
    como historial.json en su propio repositorio; los modelos Keras, como <nombre>_historial.json en
    REPO_MODELOS_KERAS. Los modelos sin entrenamiento propio (p. ej. un ensamble) no lo pasan."""
    reales = df["label"].map(etiqueta_a_id).values

    if repo_historial is not None:
        graficar_perdida(descargar_historial(repo_historial, archivo_historial), nombre)

    print(classification_report(reales, clases, target_names=etiquetas, digits=3))

    ConfusionMatrixDisplay.from_predictions(
        reales,
        clases,
        display_labels=etiquetas,
        normalize="true",  # cada fila suma 1
        values_format=".2f",
        cmap="Blues",
    )
    plt.title(f"Matriz de confusión - {nombre}")
    plt.xlabel("Clase predicha")
    plt.ylabel("Clase real")
    plt.show()

    resultados = df.assign(
        idioma=df["idioma"].astype(str),
        prediccion=[etiquetas[c] for c in clases],
        acierto=clases == reales,
    )
    display(
        resultados.groupby("idioma")["acierto"]
        .agg(accuracy="mean", resenas="size")
        .sort_values("accuracy")
    )

    errores = resultados.loc[
        ~resultados["acierto"], ["idioma", "label", "prediccion", "text"]
    ]
    print(f"{len(errores)} de {len(df)} reseñas mal clasificadas. Algunas de ellas:")
    with pd.option_context("display.max_colwidth", None):
        display(errores.sample(min(n_ejemplos, len(errores)), random_state=42))
    return errores

In [ ]:
# Cada RNN con su historial y las columnas que recibe (las mejoradas también reciben la última oración)
candidatos = [
    (nombre, historial, modelo, ("text",))
    for (nombre, historial), modelo in zip(
        historiales.items(),
        [model_rnn_1, model_rnn_2, model_rnn_3, model_rnn_4, model_rnn_5],
    )
] + [
    (f"{nombre} (mejorado)", historial, modelo, columnas_2)
    for (nombre, historial), modelo in zip(
        historiales_mejorados.items(),
        [
            model_rnn_1_mejorado,
            model_rnn_2_mejorado,
            model_rnn_3_mejorado,
            model_rnn_4_mejorado,
            model_rnn_5_mejorado,
        ],
    )
]

# La de mejor val_accuracy en la época que restauró la parada temprana
nombre_rnn, _, modelo_rnn, columnas_rnn = max(
    candidatos, key=lambda c: c[1].history["val_accuracy"][c[1].mejor_epoca]
)
print(f"Mejor RNN: {nombre_rnn}")

lotes = tf.data.Dataset.from_tensor_slices(
    entradas_modelo(test_df, columnas_rnn)
).batch(32)
clases_rnn = modelo_rnn.predict(lotes).argmax(axis=1)

# Cada RNN se construyó con name=<nombre del archivo>, así que model.name da su historial en el Hub
errores_rnn = evaluar_modelo_validacion(
    clases_rnn,
    test_df,
    nombre_rnn,
    repo_historial=REPO_MODELOS_KERAS,
    archivo_historial=f"{modelo_rnn.name}_historial.json",
)

## 4 - Transformers

Para usar transformers, usaremos PyTorch pues este puede usar GPU para correr modelos preentrenados. Comparamos tres
modelos multilingües de 12 capas que entienden las reseñas en español, inglés, francés y portugués (incluidas las
bilingües) sin tener que separarlas por idioma:

| Sección | Modelo                                          | Preentrenamiento                                               |
| ------- | ----------------------------------------------- | -------------------------------------------------------------- |
| 4.1     | `bert-base-multilingual-cased` (mBERT)          | Wikipedia en 104 idiomas                                       |
| 4.2     | `xlm-roberta-base` (XLM-RoBERTa)                | 2,5 TB de texto de CommonCrawl en 100 idiomas                  |
| 4.3     | `cardiffnlp/twitter-xlm-roberta-base-sentiment` | XLM-RoBERTa con tuits, ya ajustado para clasificar sentimiento |

A diferencia de las RNN, un transformer lee la reseña completa a la vez. Cada palabra mira a todas
las demás. Por eso puede captar por sí solo que la intención de la reseña suele estar en la última oración, sin darle esa oración como entrada aparte como en la sección 3.2.

Hacemos _fine-tuning_: sobre la salida del primer token se agrega una capa lineal de 3 salidas (negativo, neutral,
positivo) y se reentrena toda la red con una tasa de aprendizaje baja. Todo el código es común a los tres modelos y
para probar otro modelo de Hugging Face basta con pasar su nombre a `entrenar_transformer`.

**GPU en macOS y Windows.** El código elige solo el dispositivo disponible: `cuda` (GPU NVIDIA en Windows/Linux),
`mps` (GPU de Apple Silicon en macOS) o `cpu` si no hay GPU. En Windows, `pip install torch` instala por defecto la
versión **sin** CUDA: para usar la GPU hay que instalar PyTorch con el comando que indica
https://pytorch.org/get-started/locally/ (opción _CUDA_).


In [ ]:
# Instala transformers en el kernel activo (igual en macOS, Windows y Colab).
# sentencepiece lo necesita el tokenizador del modelo de Twitter (sección 4.3).
%pip install -q transformers sentencepiece

In [ ]:
import gc
import os

# En macOS, las operaciones que MPS aún no soporta se ejecutan en CPU en vez de lanzar un error.
# Debe definirse antes de importar torch.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

from types import SimpleNamespace

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


def elegir_dispositivo():
    """GPU NVIDIA (Windows/Linux) -> GPU Apple (macOS) -> CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def liberar_memoria_gpu():
    """Devuelve al sistema la memoria de la GPU que PyTorch tenía reservada y ya no usa."""
    gc.collect()
    if dispositivo.type == "cuda":
        torch.cuda.empty_cache()
    elif dispositivo.type == "mps":
        torch.mps.empty_cache()


dispositivo = elegir_dispositivo()
torch.manual_seed(42)
print(f"Dispositivo: {dispositivo}")

Reutilizamos la misma división estratificada `train_df` / `test_df` y la misma codificación de etiquetas
`etiqueta_a_id` de las secciones anteriores, para que los resultados sean comparables.

- `cargar_transformer` descarga de Hugging Face el tokenizador y el modelo, con una capa de clasificación de 3 clases.
  Si el modelo no la trae (o trae una con otro número de clases), se crea nueva y aparece un aviso de pesos
  `classifier` _MISSING_ o _MISMATCHED_, que es esperado.
- `dataset_a_torch` es el equivalente de `dataset_a_tf`: tokeniza cada lote con el tokenizador del modelo y lo
  rellena solo hasta la reseña más larga del lote. Cada modelo tiene su propio vocabulario, por eso el tokenizador se
  pasa como argumento. La reseña más larga del conjunto tiene 195 tokens con el tokenizador de mBERT, así que
  `max_length=256` no recorta ninguna.


In [ ]:
MAX_TOKENS = 256


def cargar_transformer(nombre_modelo):
    """Tokenizador y modelo `nombre_modelo` de Hugging Face, con una capa de clasificación de 3 clases."""
    tokenizador = AutoTokenizer.from_pretrained(nombre_modelo)
    torch.manual_seed(42)  # la capa nueva se inicializa igual en cada ejecución
    model = AutoModelForSequenceClassification.from_pretrained(
        nombre_modelo,
        num_labels=len(etiquetas),
        id2label=dict(enumerate(etiquetas)),
        label2id=etiqueta_a_id,
        # Si el modelo trae una capa de clasificación con otro número de clases, se reemplaza por una nueva
        ignore_mismatched_sizes=True,
        # transformers 5 carga los pesos en el tipo con que se guardó el checkpoint; el de mDeBERTa está
        # en float16, y entrenar en float16 con AdamW lleva la pérdida a NaN. Siempre entrenamos en float32.
        dtype=torch.float32,
    ).to(dispositivo)
    print(f"{nombre_modelo}: {sum(p.numel() for p in model.parameters()):,} parámetros")
    return tokenizador, model


def tokenizar(textos, tokenizador):
    return tokenizador(
        list(textos),
        padding=True,
        truncation=True,
        max_length=MAX_TOKENS,
        return_tensors="pt",
    )


def dataset_a_torch(df, shuffle, tokenizador, batch_size=16):
    """Lotes de reseñas tokenizadas con su etiqueta en `labels`."""
    ejemplos = list(zip(df["text"], df["label"].map(etiqueta_a_id)))

    def armar_lote(lote):
        textos, ids = zip(*lote)
        entradas = tokenizar(textos, tokenizador)
        entradas["labels"] = torch.tensor(ids)
        return entradas

    # num_workers=0 (valor por defecto): los procesos auxiliares del DataLoader fallan
    # dentro de notebooks en Windows y macOS, así que tokenizamos en el proceso principal.
    return DataLoader(
        ejemplos, batch_size=batch_size, shuffle=shuffle, collate_fn=armar_lote
    )

El entrenamiento sigue la misma lógica que `entrenar_modelo_con_parada_temprana`: se detiene cuando `val_loss` deja
de mejorar durante `patience` épocas y restaura los pesos de la mejor época. Devuelve un objeto con los mismos
atributos que el `History` de Keras (`history` y `mejor_epoca`), para reutilizar `resumir_historiales`.

Los transformers ya vienen preentrenados, así que bastan pocas épocas: usamos `AdamW` con tasa de aprendizaje `2e-5`,
que sube durante el primer 10 % de los pasos y luego baja linealmente a 0. Con tasas más altas el _fine-tuning_ tiende
a borrar lo aprendido en el preentrenamiento.


In [ ]:
@torch.no_grad()
def evaluar_transformer(model, loader):
    """Pérdida y accuracy promedio del modelo sobre `loader`."""
    model.eval()
    perdida, aciertos, n = 0.0, 0, 0
    for lote in loader:
        lote = lote.to(dispositivo)
        salida = model(**lote)
        perdida += salida.loss.item() * len(lote["labels"])
        aciertos += (salida.logits.argmax(dim=-1) == lote["labels"]).sum().item()
        n += len(lote["labels"])
    return perdida / n, aciertos / n


def entrenar_transformer_con_parada_temprana(
    model, train_loader, test_loader, max_epochs=4, patience=1, lr=2e-5
):
    optimizador = torch.optim.AdamW(model.parameters(), lr=lr)
    pasos = max_epochs * len(train_loader)
    planificador = get_linear_schedule_with_warmup(optimizador, int(0.1 * pasos), pasos)

    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    mejor_epoca, mejor_estado = 0, None

    for epoca in range(max_epochs):
        model.train()
        perdida, aciertos, n = 0.0, 0, 0
        for lote in tqdm(train_loader, desc=f"Época {epoca + 1}/{max_epochs}"):
            lote = lote.to(dispositivo)
            salida = model(
                **lote
            )  # con `labels`, el modelo calcula la entropía cruzada
            salida.loss.backward()
            optimizador.step()
            planificador.step()
            optimizador.zero_grad()

            perdida += salida.loss.item() * len(lote["labels"])
            aciertos += (salida.logits.argmax(dim=-1) == lote["labels"]).sum().item()
            n += len(lote["labels"])

        val_loss, val_accuracy = evaluar_transformer(model, test_loader)
        for clave, valor in zip(
            history, (perdida / n, aciertos / n, val_loss, val_accuracy)
        ):
            history[clave].append(valor)
        print(
            f"loss = {perdida / n:.4f}, accuracy = {aciertos / n:.4f}, "
            f"val_loss = {val_loss:.4f}, val_accuracy = {val_accuracy:.4f}"
        )

        if val_loss <= min(history["val_loss"]):
            # Copia de los pesos en CPU, equivalente a restore_best_weights=True
            mejor_epoca = epoca
            mejor_estado = {
                k: v.detach().cpu().clone() for k, v in model.state_dict().items()
            }
        elif epoca - mejor_epoca >= patience:
            print(f"Parada temprana en la época {epoca + 1}")
            break

    model.load_state_dict(mejor_estado)
    print(
        f"Mejor época según val_loss: {mejor_epoca + 1} "
        f"(val_loss = {history['val_loss'][mejor_epoca]:.4f}, "
        f"val_accuracy = {history['val_accuracy'][mejor_epoca]:.4f})"
    )
    return SimpleNamespace(history=history, mejor_epoca=mejor_epoca)

`entrenar_transformer` junta todos los pasos para un modelo: lo carga, lo entrena, crea su archivo submission y
predice las clases de `test_df` para la identificación del error. Para el archivo de predicciones reutilizamos
`crear_archivo_predicciones`, que llama a `model.predict` como en Keras: `PredictorTransformer` envuelve al modelo de
PyTorch con ese mismo método. Al terminar, el modelo pasa a la CPU para dejar la GPU libre para el siguiente.


In [ ]:
class PredictorTransformer:
    """Adaptador con el `predict` de Keras: recibe los lotes de textos de tf.data que arma
    `crear_archivo_predicciones` y devuelve los logits como arreglo de NumPy."""

    def __init__(self, model, tokenizador):
        self.model = model
        self.tokenizador = tokenizador

    @torch.no_grad()
    def predict(self, lotes):
        self.model.eval()
        logits = []
        for lote in lotes:
            textos = [t.decode("utf-8") for t in lote.numpy()]
            entradas = tokenizar(textos, self.tokenizador).to(dispositivo)
            logits.append(self.model(**entradas).logits.cpu())
        return torch.cat(logits).numpy()


# Los transformers son modelos de PyTorch (Hugging Face), no de Keras, así que no se guardan con
# model.save de Keras: cada uno se sube con push_to_hub a su propio repositorio privado
# <usuario>/proyecto-ml-<nombre_corto> (pesos, configuración y tokenizador), junto con historial.json,
# y se vuelve a cargar con from_pretrained.
def entrenar_o_cargar_transformer(
    nombre_modelo, nombre_corto, crear_loaders, **parametros_entrenamiento
):
    """Carga el transformer de <usuario>/proyecto-ml-<nombre_corto> si ya está en el Hub; si no,
    descarga el modelo base `nombre_modelo`, lo entrena y lo sube. `crear_loaders(tokenizador)`
    devuelve (train_loader, test_loader). Devuelve (tokenizador, modelo, historial)."""
    repo = f"{USUARIO_HF}/proyecto-ml-{nombre_corto}"

    if not REENTRENAR and api.file_exists(repo, "historial.json"):
        tokenizador = AutoTokenizer.from_pretrained(repo)
        model = AutoModelForSequenceClassification.from_pretrained(repo).to(dispositivo)
        historial = descargar_historial(repo, "historial.json")
        print(f"{nombre_corto} cargado de Hugging Face ({resumen_mejor_epoca(historial)})")
        return tokenizador, model, historial

    tokenizador, model = cargar_transformer(nombre_modelo)
    historial = entrenar_transformer_con_parada_temprana(
        model, *crear_loaders(tokenizador), **parametros_entrenamiento
    )
    model.push_to_hub(repo, private=True)
    tokenizador.push_to_hub(repo, private=True)
    # El historial se sube al final: marca que el modelo está completo
    api.upload_file(
        path_or_fileobj=historial_a_json(historial),
        path_in_repo="historial.json",
        repo_id=repo,
        commit_message="Historial de entrenamiento",
    )
    print(f"{nombre_corto} subido a https://huggingface.co/{repo}")
    return tokenizador, model, historial


def cargar_modelo_guardado(resultado):
    """Vuelve a cargar en el dispositivo el tokenizador y el modelo de `resultado` (lo que devuelve
    `entrenar_transformer`) desde Hugging Face. Después de la primera descarga quedan en la caché del
    disco, así que es rápido. Hay que liberarlo (`del`) después de usarlo."""
    repo = f"{USUARIO_HF}/proyecto-ml-{resultado.nombre_corto}"
    tokenizador = AutoTokenizer.from_pretrained(repo)
    model = AutoModelForSequenceClassification.from_pretrained(repo).to(dispositivo)
    return tokenizador, model


def probabilidades(predictor, df, columnas=("text",)):
    """Probabilidad de cada clase (softmax de los logits) para cada reseña de `df`."""
    lotes = tf.data.Dataset.from_tensor_slices(entradas_modelo(df, columnas)).batch(32)
    return torch.softmax(torch.from_numpy(predictor.predict(lotes)), dim=-1).numpy()


def resultados_transformer(
    tokenizador,
    model,
    historial,
    nombre_corto,
    test_df,
    dataset_eval,
    clase_predictor=PredictorTransformer,
    columnas=("text",),
):
    """Calcula con el modelo todo lo que usa el resto del notebook y después lo libera: las
    probabilidades en `test_df` y `dataset_eval`, las clases predichas en `test_df` y el submission.
    """
    predictor = clase_predictor(model, tokenizador)
    probs_test = probabilidades(predictor, test_df, columnas)
    probs_eval = probabilidades(predictor, dataset_eval, columnas)
    submission = subir_submission(
        probs_eval.argmax(axis=1), dataset_eval, f"submission_{nombre_corto}.csv"
    )

    del predictor, model
    liberar_memoria_gpu()
    return SimpleNamespace(
        nombre_corto=nombre_corto,
        historial=historial,
        clases_test=probs_test.argmax(axis=1),
        probs_test=probs_test,
        probs_eval=probs_eval,
        submission=submission,
    )


def entrenar_transformer(
    nombre_modelo,
    nombre_corto,
    train_df=None,
    test_df=None,
    dataset_eval=None,
    **parametros_entrenamiento,
):
    """Carga (o entrena) el transformer `nombre_modelo` y devuelve sus resultados (ver
    `resultados_transformer`): historial, clases y probabilidades en `test_df`, probabilidades en
    `dataset_eval` y submission. El modelo no queda en memoria.

    `train_df`, `test_df` y `dataset_eval` son por defecto los de las secciones anteriores; la
    sección 5 pasa `train_df_aumentado` como `train_df`. El modelo se entrena con `train_df` y se
    valida con `test_df`; `dataset_eval` (sin etiquetas) solo se usa para predecir.
    `parametros_entrenamiento` (p. ej. lr=1e-5 o max_epochs=3) se pasan a
    `entrenar_transformer_con_parada_temprana`."""
    train_df = globals()["train_df"] if train_df is None else train_df
    test_df = globals()["test_df"] if test_df is None else test_df
    dataset_eval = globals()["dataset_eval"] if dataset_eval is None else dataset_eval

    # El modelo se pasa directamente a resultados_transformer, sin guardarlo en una variable de
    # esta función, para que se libere apenas termine de usarse
    return resultados_transformer(
        *entrenar_o_cargar_transformer(
            nombre_modelo,
            nombre_corto,
            lambda tok: (
                dataset_a_torch(train_df, shuffle=True, tokenizador=tok),
                dataset_a_torch(test_df, shuffle=False, tokenizador=tok),
            ),
            **parametros_entrenamiento,
        ),
        nombre_corto,
        test_df,
        dataset_eval,
    )

### 4.1 - BERT multilingüe (mBERT)

`bert-base-multilingual-cased` es un BERT preentrenado con Wikipedia en 104 idiomas. Es nuestra referencia: los otros
dos modelos tienen la misma profundidad (12 capas), así que sus diferencias con mBERT vienen del preentrenamiento y
no de la arquitectura.


In [ ]:
mbert = entrenar_transformer(nombre_modelo="bert-base-multilingual-cased", nombre_corto="mBERT", train_df=train_df, test_df=test_df, dataset_eval= dataset_eval)

In [ ]:
mbert.submission

#### Identificación del error

Repetimos el análisis de la sección 3.3 sobre el mismo `test_df`.


In [ ]:
errores_mbert = evaluar_modelo_validacion(
    mbert.clases_test,
    test_df,
    "mBERT",
    repo_historial=f"{USUARIO_HF}/proyecto-ml-{mbert.nombre_corto}",
)

### 4.2 - XLM-RoBERTa

`xlm-roberta-base` es un RoBERTa multilingüe preentrenado con 2,5 TB de texto de CommonCrawl en 100 idiomas. Comparado
con mBERT, se preentrenó con muchos más datos por idioma (mBERT solo usó Wikipedia), con un vocabulario más grande
(250.000 piezas) y solo con la tarea de predecir palabras ocultas, por lo que suele clasificar mejor texto multilingüe.


In [ ]:
xlm_roberta = entrenar_transformer(nombre_modelo="xlm-roberta-base", nombre_corto="XLM-RoBERTa", train_df=train_df, test_df=test_df, dataset_eval=dataset_eval)

In [ ]:
xlm_roberta.submission

#### Identificación del error

Repetimos el análisis de la sección 3.3 sobre el mismo `test_df`.


In [ ]:
errores_xlm_roberta = evaluar_modelo_validacion(
    xlm_roberta.clases_test,
    test_df,
    "XLM-RoBERTa",
    repo_historial=f"{USUARIO_HF}/proyecto-ml-{xlm_roberta.nombre_corto}",
)

### 4.3 - XLM-RoBERTa ajustado para sentimiento en Twitter

`cardiffnlp/twitter-xlm-roberta-base-sentiment` es XLM-RoBERTa con un preentrenamiento adicional en unos 198 millones
de tuits multilingües, ajustado después para clasificar sentimiento en ocho idiomas, entre ellos español, inglés,
francés y portugués. Sus tres clases (_negative_, _neutral_, _positive_) están en el mismo orden que las nuestras, así
que se reutiliza su capa de clasificación ya entrenada (no aparece el aviso _MISSING_) y el _fine-tuning_ parte de un
modelo que ya distingue sentimientos. La pregunta es si lo aprendido con tuits, que son textos cortos e informales,
sirve para reseñas de productos.


In [ ]:
twitter = entrenar_transformer(nombre_modelo="cardiffnlp/twitter-xlm-roberta-base-sentiment", nombre_corto="XLM-RoBERTa-Twitter", train_df=train_df, test_df=test_df, dataset_eval=dataset_eval)

In [ ]:
twitter.submission

#### Identificación del error

Repetimos el análisis de la sección 3.3 sobre el mismo `test_df`.


In [ ]:
errores_twitter = evaluar_modelo_validacion(
    twitter.clases_test,
    test_df,
    "XLM-RoBERTa-Twitter",
    repo_historial=f"{USUARIO_HF}/proyecto-ml-{twitter.nombre_corto}",
)

### 4.4 - Comparación de todos los modelos

Comparamos los tres transformers con todas las RNN usando `resumir_historiales`. Después contamos cuántas reseñas de
`test_df` falla cada transformer y cuántas de ellas también falla la mejor RNN (sección 3.3). Si todos los modelos
fallan casi las mismas reseñas, el problema probablemente está en esas reseñas (por ejemplo, etiquetas ambiguas) y no
en la arquitectura.


In [ ]:
transformers_entrenados = {
    "mBERT": mbert,
    "XLM-RoBERTa": xlm_roberta,
    "XLM-RoBERTa-Twitter": twitter,
}

pd.concat(
    [
        resumir_historiales(historiales),
        resumir_historiales(historiales_mejorados).assign(
            modelo=lambda d: d["modelo"] + " (mejorado)"
        ),
        resumir_historiales(
            {n: r.historial for n, r in transformers_entrenados.items()}
        ),
    ]
).sort_values("val_accuracy", ascending=False)

In [ ]:
# Índices de las reseñas de test_df mal clasificadas por cada modelo
fallas_rnn = set(errores_rnn.index)
fallas = {
    "mBERT": set(errores_mbert.index),
    "XLM-RoBERTa": set(errores_xlm_roberta.index),
    "XLM-RoBERTa-Twitter": set(errores_twitter.index),
}

print(f"La mejor RNN falla {len(fallas_rnn)} reseñas.")
print(
    f"Reseñas que fallan la mejor RNN y los tres transformers: {len(fallas_rnn.intersection(*fallas.values()))}"
)

pd.DataFrame(
    {
        "errores": {n: len(f) for n, f in fallas.items()},
        "también los falla la RNN": {n: len(f & fallas_rnn) for n, f in fallas.items()},
        "solo los falla el transformer": {
            n: len(f - fallas_rnn) for n, f in fallas.items()
        },
    }
)

### 4.5 - RoBERTa con la última oración explícita


In [ ]:
def tokenizar_con_ultima(textos, ultimas, tokenizador):
    """Tokeniza pares (última oración, reseña). El tokenizador los une como
    [CLS] última oración [SEP] reseña [SEP], así el modelo ve la última oración por separado.
    """
    return tokenizador(
        list(ultimas),
        list(textos),
        padding=True,
        truncation="only_second",  # si hiciera falta recortar, se recorta la reseña, no la última oración
        max_length=MAX_TOKENS,
        return_tensors="pt",
    )


def dataset_a_torch_con_ultima(df, shuffle, tokenizador, batch_size=16):
    """Igual que `dataset_a_torch`, pero cada ejemplo es el par (última oración, reseña)."""
    ejemplos = list(zip(df["text"], df["ultima"], df["label"].map(etiqueta_a_id)))

    def armar_lote(lote):
        textos, ultimas, ids = zip(*lote)
        entradas = tokenizar_con_ultima(textos, ultimas, tokenizador)
        entradas["labels"] = torch.tensor(ids)
        return entradas

    return DataLoader(
        ejemplos, batch_size=batch_size, shuffle=shuffle, collate_fn=armar_lote
    )


class PredictorTransformerConUltima(PredictorTransformer):
    """Como `PredictorTransformer`, pero recibe los lotes {"text", "ultima"} que arma
    `crear_archivo_predicciones` con `columnas=columnas_2`."""

    @torch.no_grad()
    def predict(self, lotes):
        self.model.eval()
        logits = []
        for lote in lotes:
            textos = [t.decode("utf-8") for t in lote["text"].numpy()]
            ultimas = [t.decode("utf-8") for t in lote["ultima"].numpy()]
            entradas = tokenizar_con_ultima(textos, ultimas, self.tokenizador).to(
                dispositivo
            )
            logits.append(self.model(**entradas).logits.cpu())
        return torch.cat(logits).numpy()


def entrenar_transformer_con_ultima(
    nombre_modelo, nombre_corto, **parametros_entrenamiento
):
    """Igual que `entrenar_transformer`, pero el modelo recibe la última oración como entrada aparte."""
    return resultados_transformer(
        *entrenar_o_cargar_transformer(
            nombre_modelo,
            nombre_corto,
            lambda tok: (
                dataset_a_torch_con_ultima(train_df, shuffle=True, tokenizador=tok),
                dataset_a_torch_con_ultima(test_df, shuffle=False, tokenizador=tok),
            ),
            **parametros_entrenamiento,
        ),
        nombre_corto,
        test_df,
        dataset_eval,
        clase_predictor=PredictorTransformerConUltima,
        columnas=columnas_2,
    )

In [ ]:
roberta_ultima = entrenar_transformer_con_ultima(
    "xlm-roberta-base", "XLM-RoBERTa_ultima"
)

#### Identificación del error


In [ ]:
errores_roberta_ultima = evaluar_modelo_validacion(
    roberta_ultima.clases_test,
    test_df,
    "XLM-RoBERTa + última oración",
    repo_historial=f"{USUARIO_HF}/proyecto-ml-{roberta_ultima.nombre_corto}",
)

# Comparación con el mismo modelo sin la última oración
resumir_historiales(
    {
        "XLM-RoBERTa": xlm_roberta.historial,
        "XLM-RoBERTa+ última oración": roberta_ultima.historial,
    }
)

### 4.6 - RoBERTa Large

Finalmente entrenaremos el transformer RoBERTa Large sobre los datos sin aumentar. Asimismo, entrenaremos RoBERTa large con todos los datos de train, pues es probable que este sea el candidato al mejor modelo.

In [ ]:
xlm_roberta_large = entrenar_transformer(nombre_modelo="xlm-roberta-large", nombre_corto="XLM-RoBERTa-large", train_df=train_df, test_df=test_df, dataset_eval=dataset_eval)

In [ ]:
errores_roberta_large = evaluar_modelo_validacion(
    xlm_roberta_large.clases_test,
    test_df,
    "XLM-RoBERTa large",
    repo_historial=f"{USUARIO_HF}/proyecto-ml-{xlm_roberta_large.nombre_corto}",
)

### 4.7 - Modelo de ensemble con RoBERTa

Vemos que todos los modelos tienen distintos desempeños en distintas reseñas, y que los transformers tienen desempeño distinto en distintos idiomas. Por ejemplo RoBERTa tiene un muy buen desempeño en inglés (0.9102564102564102) y uno menor en reseñas de español-inglés (0.7878787878787878) mientras que RoBERTa tiene un muy buen desempeño en reseñas de español-inglés y un peor desempeño en reseñas en inglés (0.8888888888888888). La idea es combinar y ajustar los pesos según idioma a un modelo de ensamble 


El ensamble combina los tres RoBERTa (4.2, 4.3 y 4.5) con **votación suave ponderada por idioma**:

1. Cada modelo da la probabilidad de cada clase (softmax de sus logits) para cada reseña.
2. El peso de cada modelo en una reseña es su accuracy en `test_df` **en el idioma de esa reseña** (columna `idioma`
   de la sección 2).
3. La predicción es la clase con mayor promedio ponderado de probabilidades.

Dos detalles:

- **Idiomas con pocas reseñas.** En `test_df` hay solo 33 reseñas `es_en`, así que un solo error mueve su accuracy 3
  puntos. Para que el peso no dependa de unas pocas reseñas, el accuracy de cada idioma se suaviza hacia el accuracy
  global del modelo, como si cada idioma tuviera `PREVIA` reseñas adicionales con el accuracy global. Los idiomas
  que no aparecen en `test_df` (p. ej. reseñas sin idioma identificado) usan directamente el accuracy global.
- **Evaluación honesta.** Los pesos se calculan con `test_df`, así que medir el ensamble en ese mismo `test_df`
  sobreestimaría su accuracy. Lo evaluamos con validación cruzada: `test_df` se divide en 5 partes, los pesos se
  calculan con 4 y el ensamble se evalúa en la quinta, rotando las 5 partes. El submission final usa los pesos
  calculados con todo `test_df`.

In [ ]:
import numpy as np

# Los tres RoBERTa del ensamble. Sus probabilidades en test_df y dataset_eval ya las calculó
# entrenar_transformer (probs_test y probs_eval), así que no hace falta volver a cargar los modelos.
modelos_ensamble = {
    "XLM-RoBERTa": xlm_roberta,
    "XLM-RoBERTa-Twitter": twitter,
    "XLM-RoBERTa_ultima": roberta_ultima,
}

probs_test = {nombre: r.probs_test for nombre, r in modelos_ensamble.items()}
probs_eval = {nombre: r.probs_eval for nombre, r in modelos_ensamble.items()}

In [ ]:
PREVIA = 50  # reseñas "imaginarias" con el accuracy global que se suman a cada idioma al suavizar


def pesos_por_idioma(probs, df, previa=PREVIA):
    """Tabla idioma x modelo con el accuracy (suavizado) de cada modelo en cada idioma de `df`.

    El accuracy de cada idioma se suaviza hacia el accuracy global del modelo, como si el idioma
    tuviera `previa` reseñas más con el accuracy global; así los idiomas con pocas reseñas no
    dependen de unos pocos aciertos. La fila "otro" es el accuracy global, para los idiomas que
    no aparecen en `df`."""
    reales = df["label"].map(etiqueta_a_id).values
    idiomas = df["idioma"].astype(str).values
    tabla = {}
    for nombre, p in probs.items():
        aciertos = pd.Series(p.argmax(axis=1) == reales)
        accuracy_global = aciertos.mean()
        por_idioma = aciertos.groupby(idiomas).agg(["sum", "size"])
        suavizado = (por_idioma["sum"] + previa * accuracy_global) / (
            por_idioma["size"] + previa
        )
        tabla[nombre] = pd.concat([suavizado, pd.Series({"otro": accuracy_global})])
    return pd.DataFrame(tabla)


def votacion_suave(probs, idiomas, pesos):
    """Clase ganadora de cada reseña: promedio de las probabilidades de los modelos, ponderado por
    el peso de cada modelo (columna de `pesos`) en el idioma de la reseña (fila de `pesos`)."""
    idiomas = pd.Series(np.asarray(idiomas)).astype(str)
    idiomas = idiomas.where(idiomas.isin(pesos.index), "otro")
    w = pesos.loc[idiomas.values]  # una fila de pesos por reseña
    total = sum(w[nombre].values[:, None] * p for nombre, p in probs.items())
    return (total / w.sum(axis=1).values[:, None]).argmax(axis=1)


# Pesos calculados con todo test_df (son los que usa el submission final)
pesos_por_idioma(probs_test, test_df)

#### Evaluación del ensamble

Comparamos, sobre `test_df`, cada RoBERTa por separado, la votación suave con pesos iguales (no se ajusta nada, así
que no necesita validación cruzada) y la votación suave con pesos por idioma evaluada con validación cruzada. Como en
la sección 3.2, con ~2.000 reseñas el error estándar del accuracy es de casi 1 punto.

In [ ]:
from sklearn.model_selection import StratifiedKFold

reales_test = test_df["label"].map(etiqueta_a_id).values
idiomas_test = test_df["idioma"].astype(str).values


def evaluar_ensamble(probs_test, nombre):
    """Compara sobre `test_df` cada modelo de `probs_test` por separado, la votación suave con pesos
    iguales y la votación suave con pesos por idioma, y analiza los errores de esta última.

    Los pesos por idioma se calculan con `test_df`, así que se evalúan con validación cruzada: se
    calculan con 4/5 de `test_df` y se evalúan en el 1/5 restante. Las partes se estratifican por
    idioma para que todas tengan reseñas de cada idioma. Devuelve la tabla de accuracy, las clases
    predichas en `test_df` (validación cruzada) y las reseñas mal clasificadas."""
    clases_cv = np.empty(len(test_df), dtype=int)
    partes = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for idx_pesos, idx_evaluar in partes.split(test_df, idiomas_test):
        pesos = pesos_por_idioma(
            {n: p[idx_pesos] for n, p in probs_test.items()}, test_df.iloc[idx_pesos]
        )
        clases_cv[idx_evaluar] = votacion_suave(
            {n: p[idx_evaluar] for n, p in probs_test.items()},
            idiomas_test[idx_evaluar],
            pesos,
        )

    # Con una sola fila "otro" de unos, todos los modelos pesan lo mismo en todos los idiomas
    pesos_iguales = pd.DataFrame(1.0, index=["otro"], columns=list(probs_test))

    comparacion = pd.Series(
        {
            **{n: (p.argmax(axis=1) == reales_test).mean() for n, p in probs_test.items()},
            "Votación suave, pesos iguales": (
                votacion_suave(probs_test, idiomas_test, pesos_iguales) == reales_test
            ).mean(),
            "Votación suave, pesos por idioma (validación cruzada)": (
                clases_cv == reales_test
            ).mean(),
        },
        name="accuracy en test_df",
    )
    display(comparacion.sort_values(ascending=False).to_frame())

    errores = evaluar_modelo_validacion(clases_cv, test_df, nombre)
    return SimpleNamespace(comparacion=comparacion, clases_test=clases_cv, errores=errores)


ensamble_base = evaluar_ensamble(probs_test, "Ensamble RoBERTa por idioma")

Vemos que en este caso, no mejoró el accuracy al distinguir por idioma, y que el ensable funciona igual de bien de una u otra forma. Esto nos sugiere que el verdadero problema viene de que algunos idiomas están subrepresentados en los datos, pese a que nuestra división train/test está estratificada por idioma. 

Queda ver qué podemos ganar al introducir técnicas de aumentación de datos para que todos los idiomas estén correctamente representados.

#### Submission del ensamble

Para `dataset_eval` usamos los pesos calculados con todo `test_df` y el idioma identificado en la sección 2.

In [ ]:
def subir_submission_ensamble(probs_test, probs_eval, archivo_salida):
    """Sube el submission del ensamble con votación suave ponderada por idioma. Los pesos se
    calculan con todo `test_df` y el idioma de cada reseña de `dataset_eval` es el identificado en
    la sección 2."""
    pesos = pesos_por_idioma(probs_test, test_df)
    clases_eval = votacion_suave(probs_eval, dataset_eval["idioma"], pesos)
    return subir_submission(clases_eval, dataset_eval, archivo_salida)


submission_ensamble = subir_submission_ensamble(
    probs_test, probs_eval, "submission_ensamble_RoBERTa_idioma.csv"
)

### 4.8 -  Ensamble RoBERTa-large

Repetimos el ensamble de la sección 4.7 agregando XLM-RoBERTa large (4.6) a los tres RoBERTa base, con las mismas
funciones: `evaluar_ensamble` compara en `test_df` cada modelo, la votación suave con pesos iguales y la votación suave
con pesos por idioma (con validación cruzada), y `subir_submission_ensamble` sube el submission con los pesos
calculados con todo `test_df`.

Large tiende a inclinarse hacia positivo en las reseñas dudosas (predice 48 % de positivas en `dataset_eval`) y
XLM-RoBERTa hacia negativo (36 %), así que al promediar sus probabilidades esos sesgos pueden compensarse.

In [ ]:
# Los tres RoBERTa base de la sección 4.7 más XLM-RoBERTa large (4.6). Sus probabilidades ya las
# calculó entrenar_transformer, así que no hace falta volver a cargar ningún modelo.
probs_test_large = {**probs_test, "XLM-RoBERTa-large": xlm_roberta_large.probs_test}
probs_eval_large = {**probs_eval, "XLM-RoBERTa-large": xlm_roberta_large.probs_eval}

# Pesos de cada modelo en cada idioma, calculados con todo test_df (los que usa el submission)
display(pesos_por_idioma(probs_test_large, test_df))

ensamble_large = evaluar_ensamble(probs_test_large, "Ensamble RoBERTa + large por idioma")

submission_ensamble_large = subir_submission_ensamble(
    probs_test_large, probs_eval_large, "submission_ensamble_RoBERTa_large_idioma.csv "
)

### 4.10 - mDeBERTa-v3-base

Todos los transformers anteriores, salvo mBERT, son variantes de XLM-RoBERTa, y en la sección 4.9 vimos que se equivocan
prácticamente en las mismas reseñas. Probamos `microsoft/mdeberta-v3-base`, una familia distinta:

- **Arquitectura.** DeBERTa separa el contenido de cada palabra de su posición, y la atención entre dos palabras combina
  ambos por separado (*disentangled attention*), en lugar de sumarlos en un solo vector como RoBERTa.
- **Preentrenamiento.** En lugar de adivinar palabras enmascaradas, como XLM-RoBERTa, aprende a detectar qué palabras
  fueron reemplazadas por un generador (*replaced token detection*, como ELECTRA), y usa todas las palabras de la
  oración para aprender, no solo las enmascaradas.
- **Tamaño.** 12 capas y vectores de 768, como XLM-RoBERTa-base: 86 M de parámetros en el codificador más 190 M en los
  embeddings de su vocabulario de 250.000 tokens. Se preentrenó con CC100, los mismos datos multilingües que
  XLM-RoBERTa, así que la diferencia está en la arquitectura y en cómo se preentrenó.

Lo entrenamos con la misma función y los mismos parámetros que los demás transformers. mDeBERTa es inestable en float16, y su checkpoint está guardado en ese tipo: `transformers` 5 lo cargaba en float16 y
la pérdida llegaba a `NaN` desde el segundo paso de entrenamiento. Por eso `cargar_transformer` fuerza `float32`.

In [ ]:
mdeberta = entrenar_transformer(nombre_modelo="microsoft/mdeberta-v3-base", nombre_corto="mDeBERTa", train_df=train_df, test_df=test_df, dataset_eval=dataset_eval)

In [ ]:
errores_mdeberta = evaluar_modelo_validacion(
    mdeberta.clases_test,
    test_df,
    "mDeBERTa",
    repo_historial=f"{USUARIO_HF}/proyecto-ml-{mdeberta.nombre_corto}",
)

#### Comparación con XLM-RoBERTa

Además de comparar sus métricas, vemos si mDeBERTa se equivoca en reseñas distintas que XLM-RoBERTa: si sus errores
coinciden casi por completo, combinarlos tampoco ayudaría.

In [ ]:
display(
    resumir_historiales(
        {
            "XLM-RoBERTa": xlm_roberta.historial,
            "XLM-RoBERTa-large": xlm_roberta_large.historial,
            "mDeBERTa": mdeberta.historial,
        }
    )
)

# Reseñas de test_df según qué modelo acierta
acierta_xlm = pd.Series(xlm_roberta.clases_test == reales_test, name="XLM-RoBERTa acierta")
acierta_mdeberta = pd.Series(mdeberta.clases_test == reales_test, name="mDeBERTa acierta")
display(pd.crosstab(acierta_xlm, acierta_mdeberta, margins=True, margins_name="total"))
print(
    f"Errores de mDeBERTa que XLM-RoBERTa no comete: {(~acierta_mdeberta & acierta_xlm).sum()}; "
    f"errores de XLM-RoBERTa que mDeBERTa no comete: {(~acierta_xlm & acierta_mdeberta).sum()}; "
    f"errores de ambos: {(~acierta_xlm & ~acierta_mdeberta).sum()}"
)

## 5 - Aumentación de datos

Ahora, intentaremos hacer aumentación de datos para reentrenar los datos y aumentar el accuracy, especialmente in idiomas subrepresentados (francés, español-ingles o portugués). Vemos que las reseñas neutras ya están perfectamente predichas, por lo que enfocaremos la aumentación de datos en generar reseñas positivas y negativas. 

Asimismo, como dijimos antes, los idiomas no están representados equitativamente y en el archivo de evaluación, la distribución de lenguaje no corresponde a la distribución de lenguaje en el archivo de entrenamiento.

Asimismo, la mayoría de reseñas que fallan lo hacen en reseñas mixtas, en las que en general la última oración no decide la intención de la reseña. En teoría los transformers pueden asociar la polaridad de la última oración con la intención total de la reseña, pero como los idiomas estaban subrepresentados, la identificación de esto no era suficiente.

### 5.1 - Estrategia 1: intercambio de opiniones en reseñas mixtas

**Dónde están los errores.** Analizamos las 205 reseñas de `test_df` que XLM-RoBERTa (4.2) clasifica mal:

- Las neutras ya se predicen perfecto: todos los errores son confusiones entre positivo y negativo.
- Cuando el modelo está seguro acierta casi siempre: con probabilidad de al menos 0,95 su accuracy es 99,6 %.
- 154 de los 205 errores tienen probabilidad menor que 0,7, y en esa zona el modelo acierta solo el 51 %: es una
  moneda al aire. En `dataset_eval` el 17 % de las reseñas cae en esa zona (15 % en `test_df`).

Esas reseñas son casi siempre **mixtas**: relleno sobre la compra y el envío, un aspecto que se elogia y otro que se
critica, y a veces una frase de cierre ambigua ("cada quien que juzgue", "ni fu ni fa"). Para saber qué decide su
etiqueta, puntuamos cada oración por separado con XLM-RoBERTa y nos quedamos con las reseñas que tienen exactamente una
opinión positiva y una negativa.

**La regla: gana la última opinión, según el conector.** En el 86 % de esas reseñas la etiqueta es la polaridad de la
última opinión y solo en el 14 % la de la primera. Que la regla se cumpla depende de la palabra con que empieza la última
opinión (la tabla completa se calcula en la celda siguiente):

| Conector de la última opinión (español)              | La última opinión gana |
| ---------------------------------------------------- | ---------------------- |
| _aun así_, _con todo_, _al final_, _sin embargo_     | 98 % a 100 %           |
| _eso sí_                                             | 91 %                   |
| sin conector                                         | 89 %                   |
| _ahora_, _además_                                    | 67 % a 79 %            |
| _por otro lado_, _de paso_, _por cierto_             | 55 % a 57 %            |

En inglés, francés y portugués la regla es más débil (69 % a 81 % sin conector). También probamos si el aspecto
(batería, envío, pantalla...) o las palabras de cada opinión ayudan a predecir la etiqueta: con una regresión logística
y validación cruzada el accuracy queda igual (0,86). Es decir, en las reseñas que no cumplen la regla la etiqueta no
depende de nada que esté en el texto: se comportan como ruido de las etiquetas, y ningún modelo las va a acertar más
que por azar. Lo que sí se puede ganar son las reseñas que cumplen la regla y el modelo igual falla.

**La aumentación.** El modelo tiende a asociar aspectos y palabras con una etiqueta en lugar de fijarse en la posición
y el conector. Para enseñarle la regla generamos **pares contrafactuales**: tomamos una reseña mixta, intercambiamos sus
dos opiniones dejando cada conector en su lugar e invertimos la etiqueta.

> **Original (negativo, abreviada):** _El cargador es justo lo que estaba buscando. Eso sí, la señal no vale lo que cuesta._
>
> **Intercambiada (positivo, abreviada):** _La señal no vale lo que cuesta. Eso sí, el cargador es justo lo que estaba buscando._

Las dos reseñas tienen exactamente las mismas palabras y etiquetas opuestas, así que la única manera de acertar ambas
es aprender que decide la opinión que va después del conector. Para no agregar etiquetas incorrectas:

1. Una oración es una opinión si XLM-RoBERTa le da al menos 0,9 de probabilidad a positivo o a negativo
   (`UMBRAL_OPINION`).
2. Solo se usan reseñas con exactamente una opinión positiva y una negativa.
3. Solo se intercambian los pares (idioma, conector) en los que la regla se cumple al menos el 90 % de las veces en
   al menos 20 reseñas de entrenamiento (`UMBRAL_REGLA`, `MIN_RESENAS_CONECTOR`). Los conectores se comparan en
   minúsculas y sin tildes, así _eso sí_ y _eso si_ cuentan como uno.
4. Solo se intercambian reseñas cuya etiqueta cumple la regla, para no multiplicar las que se comportan como ruido.
5. Solo se usa `train_df`: `test_df` queda intacto para medir la mejora de forma honesta.

**Qué no hacemos.** Mezclar las oraciones al azar o borrarlas (técnicas típicas de aumentación) destruiría justamente
la señal de la que depende la etiqueta, así que bajaría el accuracy.

**Limitación.** Con estos umbrales solo pasan conectores del español, porque en los otros idiomas la regla no es
confiable. Esta aumentación ataca las reseñas mixtas, no la diferencia de idiomas entre `train_df` y `dataset_eval`.

**Cómo medir la mejora.** Reentrenamos el mismo transformer con `train_df_aumentado` y comparamos su accuracy en
`test_df` con el original. Con 2.047 reseñas el error estándar del accuracy es de unos 0,66 puntos, así que una mejora
de 2 puntos se puede distinguir del azar. Sin etiquetas de `dataset_eval`, la mejor señal ahí es que baje la fracción de
reseñas con probabilidad menor que 0,7.

In [ ]:
import re
import unicodedata
import numpy as np

# Una oración expresa una opinión si el transformer le da al menos esta probabilidad a positivo o a negativo
UMBRAL_OPINION = 0.9
# Solo se intercambian reseñas cuyo conector cumple la regla "gana la última opinión" al menos así de seguido...
UMBRAL_REGLA = 0.9
# ...medido en al menos esta cantidad de reseñas de entrenamiento
MIN_RESENAS_CONECTOR = 20


def separar_oraciones(texto):
    """Las mismas oraciones que usa `ultima_oracion` (corta después de . ! ?)."""
    return [o for o in re.split(r"(?<=[.!?])\s+", texto.strip()) if o]


def polaridad_oraciones(resultado, oraciones, batch_size=64):
    """Probabilidad (negativo, neutral, positivo) de cada oración por separado, con un transformer
    ya ajustado (el resultado de `entrenar_transformer`). El modelo se vuelve a cargar de Hugging Face
    solo para esta predicción y se libera al terminar."""
    lotes = tf.data.Dataset.from_tensor_slices(list(oraciones)).batch(batch_size)
    tokenizador, model = cargar_modelo_guardado(resultado)
    logits = PredictorTransformer(model, tokenizador).predict(lotes)
    del model
    liberar_memoria_gpu()
    return torch.softmax(torch.from_numpy(logits), dim=-1).numpy()


def separar_conector(oracion):
    """('Aun así', 'el teclado me resultó sólido.') si la oración empieza con 1 a 3 palabras
    seguidas de una coma; ('', oracion) si no."""
    m = re.match(r"^((?:[^\W\d_]+['’]?\s?){1,3}),\s+(.+)$", oracion, re.S)
    return (m.group(1).strip(), m.group(2)) if m else ("", oracion)


def normalizar_conector(conector):
    """Minúsculas y sin tildes, para que "Eso sí" y "Eso si" cuenten como el mismo conector."""
    sin_tildes = unicodedata.normalize("NFKD", conector.lower())
    return "".join(c for c in sin_tildes if not unicodedata.combining(c))


def unir_conector(conector, resto):
    """Arma la oración con el conector de esa posición, ajustando la mayúscula inicial."""
    if not conector:
        return resto[:1].upper() + resto[1:]
    if not re.match(r"^I\b", resto):  # el pronombre "I" del inglés va siempre en mayúscula
        resto = resto[:1].lower() + resto[1:]
    return f"{conector}, {resto}"


def opiniones_mixtas(df, resultado):
    """Una fila por cada reseña positiva o negativa de `df` con exactamente una oración claramente
    positiva y una claramente negativa: sus oraciones, la posición de cada opinión y el conector
    (normalizado) con que empieza la última opinión."""
    df = df[df["label"] != "neutral"]
    oraciones = df["text"].map(separar_oraciones)
    planas = [o for lista in oraciones for o in lista]
    probs = polaridad_oraciones(resultado, planas)

    filas, inicio = [], 0
    for idx, lista in oraciones.items():
        p = probs[inicio : inicio + len(lista)]
        inicio += len(lista)
        pos = [k for k in range(len(lista)) if p[k, 2] >= UMBRAL_OPINION]
        neg = [k for k in range(len(lista)) if p[k, 0] >= UMBRAL_OPINION]
        if len(pos) == 1 and len(neg) == 1:
            i, j = sorted((pos[0], neg[0]))  # i: primera opinión, j: última opinión
            filas.append(
                {
                    "indice": idx,
                    "oraciones": lista,
                    "i": i,
                    "j": j,
                    "polaridad_ultima": "positivo" if j == pos[0] else "negativo",
                    "conector": normalizar_conector(separar_conector(lista[j])[0]),
                }
            )
    mixtas = pd.DataFrame(filas).set_index("indice")
    return mixtas.join(df[["id", "label", "idioma"]])


def confiabilidad_conectores(mixtas):
    """Para cada idioma y conector de la última opinión: cuántas reseñas hay y en qué fracción
    de ellas la etiqueta coincide con la polaridad de la última opinión."""
    cumple = mixtas["label"] == mixtas["polaridad_ultima"]
    tabla = cumple.groupby(
        [mixtas["idioma"].astype(str), mixtas["conector"].replace("", "(sin conector)")]
    ).agg(resenas="size", ultima_gana="mean")
    tabla["se_intercambia"] = (tabla["resenas"] >= MIN_RESENAS_CONECTOR) & (
        tabla["ultima_gana"] >= UMBRAL_REGLA
    )
    return tabla.sort_values("resenas", ascending=False)


def intercambiar_opiniones(oraciones, i, j, conectores):
    """Intercambia las opiniones de las posiciones i < j. Los conectores se quedan en su
    posición, así "A. Aun así, B." pasa a ser "B. Aun así, A."; solo se separan como conector
    los que están en `conectores`, para no mover por error el comienzo de la opinión."""

    def partir(oracion):
        conector, resto = separar_conector(oracion)
        return (conector, resto) if normalizar_conector(conector) in conectores else ("", oracion)

    con_i, op_i = partir(oraciones[i])
    con_j, op_j = partir(oraciones[j])
    nuevas = list(oraciones)
    nuevas[i] = unir_conector(con_i, op_j)
    nuevas[j] = unir_conector(con_j, op_i)
    return " ".join(nuevas)


def generar_intercambios(mixtas, tabla):
    """Reseñas nuevas: las opiniones intercambiadas y la etiqueta invertida. Solo se usan las
    reseñas cuyo (idioma, conector) es confiable y cuya etiqueta cumple la regla, para no
    multiplicar el ruido de las que no la cumplen."""
    confiables = set(tabla.index[tabla["se_intercambia"]])
    conectores = {c for _, c in tabla.index[tabla["resenas"] >= MIN_RESENAS_CONECTOR]}
    conectores.discard("(sin conector)")

    clave = list(
        zip(
            mixtas["idioma"].astype(str),
            mixtas["conector"].replace("", "(sin conector)"),
        )
    )
    usar = mixtas[
        pd.Series(clave, index=mixtas.index).isin(confiables)
        & (mixtas["label"] == mixtas["polaridad_ultima"])
    ]
    invertir = {"positivo": "negativo", "negativo": "positivo"}
    nuevas = pd.DataFrame(
        {
            "id": usar["id"],
            "text": [
                intercambiar_opiniones(o, i, j, conectores)
                for o, i, j in zip(usar["oraciones"], usar["i"], usar["j"])
            ],
            "label": usar["label"].map(invertir),
            "idioma": usar["idioma"],
        }
    )
    nuevas["ultima"] = nuevas["text"].map(ultima_oracion)
    nuevas["origen"] = nuevas.index  # índice de la reseña original en train_df
    return nuevas.reset_index(drop=True)


# Las opiniones se detectan con el XLM-RoBERTa de la sección 4.2. Solo se usa train_df: test_df
# queda intacto para medir honestamente si la aumentación mejora el accuracy.
mixtas = opiniones_mixtas(train_df, xlm_roberta)
tabla_conectores = confiabilidad_conectores(mixtas)
print(f"{len(mixtas)} reseñas de train_df con una opinión positiva y una negativa")
display(tabla_conectores.head(20))

intercambios = generar_intercambios(mixtas, tabla_conectores)
print(
    f"{len(intercambios)} reseñas nuevas por intercambio "
    f"({len(intercambios) / len(train_df):.1%} de train_df)"
)
print(intercambios["label"].value_counts().to_dict(), intercambios["idioma"].value_counts().to_dict())

# Algunos pares (original, intercambiada) para revisar a mano
with pd.option_context("display.max_colwidth", None):
    ejemplos = intercambios.sample(5, random_state=42)
    display(
        pd.DataFrame(
            {
                "original": train_df.loc[ejemplos["origen"], "text"].values,
                "etiqueta original": train_df.loc[ejemplos["origen"], "label"].values,
                "intercambiada": ejemplos["text"].values,
                "etiqueta nueva": ejemplos["label"].values,
            }
        )
    )

# train_df + intercambios, con índices nuevos para las reseñas generadas
train_df_aumentado = pd.concat(
    [
        train_df,
        intercambios.drop(columns="origen").set_axis(
            train_df.index.max() + 1 + np.arange(len(intercambios))
        ),
    ]
)
print(f"train_df_aumentado: {len(train_df_aumentado)} reseñas")

In [ ]:
xlm_roberta_aumentado =  entrenar_transformer(nombre_modelo="xlm-roberta-base", nombre_corto="XLM-RoBERTa_aumentado", train_df=train_df_aumentado, test_df=test_df, dataset_eval=dataset_eval)

En general, podemos ver que esta estrategia no funcionó, y en los datos de evaluación generó un accuracy de 0.876, menor que sin aumentación, y en los datos de validación fue peor que el ensamble y los transformers sin aumentación. Asimismo, vemos que las reseñas están sesgadas a positivo, por lo que podríamos intentar mover el umbral a negativo.

### 5.2 - Estrategia 2: mover el umbral entre positivo y negativo

**Por qué falló el intercambio de opiniones.** El modelo aumentado no distingue mejor las reseñas positivas de las
negativas: el AUC entre ambas clases en `test_df` es prácticamente el mismo (0,960 contra 0,959 de XLM-RoBERTa sin
aumentar). Lo que cambió fue hacia dónde se inclina en las reseñas dudosas. En `dataset_eval` pasó 1.039 reseñas de
negativo a positivo (y solo 14 al revés), 835 de ellas en la zona donde el modelo original tenía confianza menor que
0,7, y la fracción de positivas subió de 36 % a 51 %. En `test_df` el recall de negativo bajó de 0,915 a 0,730 y el de
positivo subió de 0,810 a 0,964.

Es decir, en esa zona las probabilidades de positivo y negativo son parecidas y cada entrenamiento termina inclinándose
hacia un lado de forma bastante arbitraria: XLM-RoBERTa y XLM-RoBERTa_ultima predicen más negativo; mBERT, el modelo
de Twitter y el aumentado, más positivo. Como esa zona es cerca del 17 % de `dataset_eval`, la dirección en que se
inclina el modelo mueve bastante el accuracy.

**Qué dice el leaderboard.** Los puntajes no corresponden a las 6.781 reseñas de `dataset_eval`: coinciden exactamente
con 1.815, 1.811 y 1.784 aciertos de 2.036 reseñas (0,89145, 0,88948 y 0,87622), es decir, el leaderboard público se
calcula con cerca del 30 % de `dataset_eval`. Eso tiene dos consecuencias:

- Con 2.036 reseñas el error estándar del accuracy es de unos 0,7 puntos (unas 14 reseñas). La diferencia entre
  XLM-RoBERTa y el ensamble (4 reseñas) está dentro del ruido.
- XLM-RoBERTa y el modelo aumentado se diferencian casi solo en las reseñas dudosas que el aumentado pasó a positivo, y
  el aumentado perdió 31 aciertos. Eso indica que en `dataset_eval` cerca del 55 % de esas reseñas son negativas. En
  `test_df` pasa lo mismo, aunque menos marcado: en la zona dudosa hay 164 negativas y 149 positivas.

**La técnica.** No hace falta reentrenar: cambiamos solo la regla de decisión. Para cada reseña que el modelo no
predice como neutral calculamos $r = p_{pos} / (p_{pos} + p_{neg})$ y la clasificamos como positiva solo si $r$ supera
`UMBRAL_POSITIVO`, en lugar de 0,5 como hace el argmax. Las reseñas neutras no cambian, porque ya se predicen
perfecto.

**Elección del umbral.** La celda siguiente muestra el accuracy en `test_df` con cada umbral y lo evalúa con
validación cruzada: el umbral se elige con 4/5 de `test_df` y se mide en el 1/5 restante, para no evaluar con las
mismas reseñas con que se eligió. En las cinco partes el mejor umbral es 0,55, y el accuracy sube de 0,8999 a 0,9047.
Así que `test_df` por sí solo ya indica que conviene inclinarse hacia negativo. Elegimos 0,7, un poco más inclinado,
porque el leaderboard indica que `dataset_eval` está más cargado hacia negativo que `test_df`. Con 0,7 el accuracy en
`test_df` sigue mejorando (0,9023), y desde 0,9 empieza a empeorar (0,8945).

**Qué esperamos.** Con 0,7 pasan de positivo a negativo 276 reseñas de `dataset_eval`. Unas 83 deberían caer en el
leaderboard público; si el 55 % de ellas son negativas, se ganan unos 8 aciertos y el puntaje subiría a cerca de 0,895.
Con un error estándar de 14 reseñas, el resultado puede quedar varios aciertos arriba o abajo de eso. Además, si la
competencia tiene un leaderboard privado, se calcula con el otro 70 % de `dataset_eval`, donde la proporción podría
ser distinta.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Entre positivo y negativo, solo se predice positivo si p_pos / (p_pos + p_neg) supera este umbral
UMBRAL_POSITIVO = 0.7


def clases_con_umbral(probs, umbral):
    """Igual que argmax, salvo entre positivo y negativo: la reseña es positiva solo si
    p_pos / (p_pos + p_neg) > umbral. Las reseñas que el modelo predice como neutras no cambian."""
    neg, neu, pos = (etiqueta_a_id[e] for e in ("negativo", "neutral", "positivo"))
    clases = probs.argmax(axis=1)
    polares = clases != neu
    r = probs[:, pos] / (probs[:, pos] + probs[:, neg])
    clases[polares] = np.where(r[polares] > umbral, pos, neg)
    return clases


# Probabilidades de XLM-RoBERTa (4.2) que ya calculamos para el ensamble (4.7)
p_test = probs_test["XLM-RoBERTa"]
p_eval = probs_eval["XLM-RoBERTa"]
clases_argmax_eval = p_eval.argmax(axis=1)

# 1. Qué pasa con cada umbral: accuracy en test_df y predicciones de eval que cambian
umbrales = [0.5, 0.6, 0.7, 0.8, 0.9]
display(
    pd.DataFrame(
        {
            "accuracy en test_df": [
                (clases_con_umbral(p_test, u) == reales_test).mean() for u in umbrales
            ],
            "reseñas de eval que pasan a negativo": [
                (clases_con_umbral(p_eval, u) != clases_argmax_eval).sum() for u in umbrales
            ],
        },
        index=pd.Index(umbrales, name="umbral"),
    )
)

# 2. Validación cruzada: el umbral se elige con 4/5 de test_df y se evalúa en el 1/5 restante,
# para no medir el accuracy en las mismas reseñas con que se eligió el umbral
grilla = np.round(np.arange(0.5, 0.91, 0.05), 2)
clases_cv = np.empty(len(test_df), dtype=int)
umbrales_cv = []
for idx_elegir, idx_evaluar in StratifiedKFold(
    n_splits=5, shuffle=True, random_state=42
).split(p_test, reales_test):
    mejor = max(
        grilla,
        key=lambda u: (
            clases_con_umbral(p_test[idx_elegir], u) == reales_test[idx_elegir]
        ).mean(),
    )
    umbrales_cv.append(float(mejor))
    clases_cv[idx_evaluar] = clases_con_umbral(p_test[idx_evaluar], mejor)
print(
    f"Validación cruzada en test_df: accuracy {(clases_cv == reales_test).mean():.4f} "
    f"(sin mover el umbral: {(p_test.argmax(axis=1) == reales_test).mean():.4f}); "
    f"umbrales elegidos en cada parte: {umbrales_cv}"
)

# 3. Submission con el umbral elegido
clases_eval_umbral = clases_con_umbral(p_eval, UMBRAL_POSITIVO)
print(
    f"Umbral {UMBRAL_POSITIVO}: {(clases_eval_umbral != clases_argmax_eval).sum()} reseñas de "
    "eval pasan de positivo a negativo; el resto de las predicciones no cambia"
)
submission_umbral = subir_submission(
    clases_eval_umbral,
    dataset_eval,
    f"submission_XLM-RoBERTa_umbral{round(UMBRAL_POSITIVO * 10):02d}.csv",
)

### 5.3 - Estrategia 3: traducción automática con NLLB

**Idea.** Los idiomas no tienen la misma proporción en `dataset_eval` que en `train_df`, que es 65 % español.
Traducimos reseñas positivas y negativas en español de `train_df` a los idiomas que faltan con
`facebook/nllb-200-distilled-600M`, un transformer seq2seq de Meta que traduce entre 200 idiomas. A diferencia de
la sección 5.1, la etiqueta no se cambia: una buena traducción conserva el sentimiento de la reseña.

**Decisiones**, a partir de lo que aprendimos en 5.1 y 5.2:

1. **Se traduce oración por oración** (con `separar_oraciones` de 5.1) y se vuelven a unir. NLLB tiende a omitir
   oraciones cuando recibe textos largos, y así cada conector ("aun así", "con todo") se traduce dentro de su oración.
2. **Cuántas reseñas por idioma:** las que le faltan a `train_df` para acercarse a la proporción de ese idioma en
   `dataset_eval`, con un tope de `MAX_POR_IDIOMA`. Las reseñas `es_en` no se generan, porque NLLB no produce texto
   que mezcle dos idiomas.
3. **Filtro de consistencia:** XLM-RoBERTa (4.2) clasifica cada traducción y se descartan las que le dan menos de
   `UMBRAL_CONSISTENCIA` a su etiqueta, porque probablemente la traducción perdió el sentido.
4. **Misma proporción positivo/negativo que `train_df`** en cada idioma, para no mover el sesgo de la decisión
   como pasó en 5.1.
5. **Solo se aumenta `train_df`.** `test_df` queda intacto. Las traducciones se guardan en `REPO_DATOS`
   (`aumentacion/traducciones_nllb.csv`) para no volver a generarlas.

**Cómo evaluar.** Además del accuracy, miramos el AUC entre positivo y negativo y el accuracy con el umbral elegido
por validación cruzada (5.2). Si el accuracy sube pero el AUC no, la aumentación solo movió el sesgo, como en 5.1.

In [ ]:
from transformers import AutoModelForSeq2SeqLM

NLLB = "facebook/nllb-200-distilled-600M"
CODIGO_NLLB = {"es": "spa_Latn", "en": "eng_Latn", "pt_br": "por_Latn", "fr": "fra_Latn"}
# Idiomas a los que se traduce (es_en no: NLLB no genera reseñas que mezclen dos idiomas)
IDIOMAS_DESTINO = ["en", "pt_br", "fr"]
MAX_POR_IDIOMA = 1500  # tope de reseñas traducidas por idioma, porque traducir es lento
RUTA_TRADUCCIONES = "aumentacion/traducciones_nllb.csv"
REGENERAR_TRADUCCIONES = False


@torch.no_grad()
def traducir_oraciones(oraciones, origen, destino, tokenizador, model, batch_size=32):
    """Traduce cada oración de `oraciones` del idioma `origen` a `destino` (claves de CODIGO_NLLB)."""
    tokenizador.src_lang = CODIGO_NLLB[origen]
    id_destino = tokenizador.convert_tokens_to_ids(CODIGO_NLLB[destino])
    # Ordenar por longitud reduce el padding de cada lote y acelera mucho la generación
    orden = np.argsort([len(o) for o in oraciones])
    traducidas = [None] * len(oraciones)
    for k in tqdm(range(0, len(orden), batch_size), desc=f"{origen} -> {destino}"):
        idx = orden[k : k + batch_size]
        entradas = tokenizador(
            [oraciones[i] for i in idx],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(dispositivo)
        generado = model.generate(
            **entradas, forced_bos_token_id=id_destino, num_beams=4, max_new_tokens=160
        )
        for i, texto in zip(idx, tokenizador.batch_decode(generado, skip_special_tokens=True)):
            traducidas[i] = texto
    return traducidas


def traducir_resenas(textos, origen, destino, tokenizador, model):
    """Traduce cada reseña oración por oración y vuelve a unir sus oraciones traducidas."""
    oraciones = [separar_oraciones(t) for t in textos]
    planas = [o for lista in oraciones for o in lista]
    traducidas = iter(traducir_oraciones(planas, origen, destino, tokenizador, model))
    return [" ".join(next(traducidas) for _ in lista) for lista in oraciones]


def resenas_por_idioma(train_df, dataset_eval):
    """Cuántas reseñas traducir a cada idioma de IDIOMAS_DESTINO: las que le faltan a `train_df` para
    tener (aproximadamente) la proporción de ese idioma en `dataset_eval`, con tope MAX_POR_IDIOMA."""
    prop_train = train_df["idioma"].value_counts(normalize=True)
    prop_eval = dataset_eval["idioma"].value_counts(normalize=True)
    faltan = (prop_eval.reindex(IDIOMAS_DESTINO, fill_value=0) - prop_train.reindex(IDIOMAS_DESTINO, fill_value=0))
    return (faltan * len(train_df)).clip(lower=0).round().astype(int).clip(upper=MAX_POR_IDIOMA)


def generar_traducciones(train_df, dataset_eval):
    """Traduce reseñas positivas y negativas en español de `train_df` a cada idioma de destino. La
    muestra de cada idioma es estratificada por etiqueta, así que conserva la proporción positivo/negativo."""
    polares_es = train_df[(train_df["idioma"] == "es") & (train_df["label"] != "neutral")]
    tokenizador = AutoTokenizer.from_pretrained(NLLB)
    model = AutoModelForSeq2SeqLM.from_pretrained(NLLB).to(dispositivo).eval()

    partes = []
    for k, (destino, n) in enumerate(resenas_por_idioma(train_df, dataset_eval).items()):
        if n == 0:
            continue
        muestra = polares_es.groupby("label", group_keys=False).sample(
            frac=min(n / len(polares_es), 1), random_state=42 + k
        )
        partes.append(
            pd.DataFrame(
                {
                    "origen": muestra.index,  # índice de la reseña original en train_df
                    "id": muestra["id"].values,
                    "text": traducir_resenas(muestra["text"].tolist(), "es", destino, tokenizador, model),
                    "label": muestra["label"].values,
                    "idioma": destino,
                }
            )
        )

    del model
    liberar_memoria_gpu()
    if not partes:
        raise ValueError("A ningún idioma de IDIOMAS_DESTINO le faltan reseñas respecto a dataset_eval")
    return pd.concat(partes, ignore_index=True)


display(
    pd.DataFrame(
        {
            "train_df": train_df["idioma"].value_counts(normalize=True),
            "dataset_eval": dataset_eval["idioma"].value_counts(normalize=True),
        }
    ).round(3)
)
print("Reseñas a traducir por idioma:", resenas_por_idioma(train_df, dataset_eval).to_dict())

if not REGENERAR_TRADUCCIONES and api.file_exists(REPO_DATOS, RUTA_TRADUCCIONES, repo_type="dataset"):
    traducciones = pd.read_csv(f"hf://datasets/{REPO_DATOS}/{RUTA_TRADUCCIONES}")
    print(f"{RUTA_TRADUCCIONES} ya existe en Hugging Face; no se vuelve a traducir")
else:
    traducciones = generar_traducciones(train_df, dataset_eval)
    api.upload_file(
        path_or_fileobj=traducciones.to_csv(index=False).encode(),
        path_in_repo=RUTA_TRADUCCIONES,
        repo_id=REPO_DATOS,
        repo_type="dataset",
        commit_message="Traducciones NLLB para aumentación (sección 5.3)",
    )
    print(f"{RUTA_TRADUCCIONES} subido a https://huggingface.co/datasets/{REPO_DATOS}")

display(pd.crosstab(traducciones["idioma"], traducciones["label"]))

In [ ]:
# Se descartan las traducciones a las que XLM-RoBERTa (4.2) da menos de esta probabilidad en su etiqueta
UMBRAL_CONSISTENCIA = 0.5

# Proporción de positivas entre las reseñas positivas y negativas de train_df
polares_train = train_df[train_df["label"] != "neutral"]
PROP_POSITIVO = (polares_train["label"] == "positivo").mean()


def equilibrar(df, prop_positivo, semilla=42):
    """El mayor subconjunto de `df` con la proporción `prop_positivo` de positivas entre positivas y negativas."""
    pos, neg = df[df["label"] == "positivo"], df[df["label"] == "negativo"]
    n_total = int(min(len(pos) / prop_positivo, len(neg) / (1 - prop_positivo)))
    n_pos = round(n_total * prop_positivo)
    return pd.concat(
        [pos.sample(n_pos, random_state=semilla), neg.sample(n_total - n_pos, random_state=semilla)]
    )


# 1. Filtro de consistencia (el modelo se recarga de Hugging Face solo para esta predicción)
probs_traducciones = polaridad_oraciones(xlm_roberta, traducciones["text"])
p_etiqueta = probs_traducciones[
    np.arange(len(traducciones)), traducciones["label"].map(etiqueta_a_id).values
]
consistentes = traducciones[p_etiqueta >= UMBRAL_CONSISTENCIA]

# 2. La misma proporción positivo/negativo que train_df en cada idioma
aumentadas = pd.concat([equilibrar(g, PROP_POSITIVO) for _, g in consistentes.groupby("idioma")])

display(
    pd.DataFrame(
        {
            "traducidas": traducciones["idioma"].value_counts(),
            "consistentes": consistentes["idioma"].value_counts(),
            "finales": aumentadas["idioma"].value_counts(),
            "% positivo final": aumentadas.groupby("idioma")["label"].apply(lambda s: (s == "positivo").mean()),
        }
    ).round(3)
)
print(f"% positivo entre positivas y negativas de train_df: {PROP_POSITIVO:.3f}")

# 3. train_df + traducciones, con índices nuevos para las reseñas generadas
nuevas = aumentadas.drop(columns="origen").assign(ultima=lambda d: d["text"].map(ultima_oracion))
train_df_traducido = pd.concat(
    [train_df, nuevas.set_axis(train_df.index.max() + 1 + np.arange(len(nuevas)))]
)
print(f"train_df_traducido: {len(train_df_traducido)} reseñas ({len(nuevas)} nuevas)")
display(
    pd.DataFrame(
        {
            "train_df_traducido": train_df_traducido["idioma"].value_counts(normalize=True),
            "dataset_eval": dataset_eval["idioma"].value_counts(normalize=True),
        }
    ).round(3)
)

In [ ]:
# Si se cambian las traducciones o los umbrales, hay que reentrenar: con REENTRENAR = False se carga el
# modelo que ya está en Hugging Face, aunque se haya entrenado con otro train_df_traducido.
xlm_roberta_traducido = entrenar_transformer(
    nombre_modelo="xlm-roberta-base",
    nombre_corto="XLM-RoBERTa_traducido",
    train_df=train_df_traducido,
    test_df=test_df,
    dataset_eval=dataset_eval,
)

In [ ]:
errores_roberta_traducido = evaluar_modelo_validacion(
    xlm_roberta_traducido.clases_test,
    test_df,
    "XLM-RoBERTa traducido",
    repo_historial=f"{USUARIO_HF}/proyecto-ml-{xlm_roberta_traducido.nombre_corto}",
)

In [ ]:
from sklearn.metrics import roc_auc_score


def auc_positivo_negativo(probs, df):
    """AUC de r = p_pos / (p_pos + p_neg) para separar las reseñas positivas de las negativas de `df`."""
    neg, pos = etiqueta_a_id["negativo"], etiqueta_a_id["positivo"]
    polares = (df["label"] != "neutral").values
    r = probs[polares, pos] / (probs[polares, pos] + probs[polares, neg])
    return roc_auc_score(df["label"].values[polares] == "positivo", r)


def accuracy_umbral_cv(probs, reales, grilla=np.round(np.arange(0.5, 0.91, 0.05), 2)):
    """Accuracy con el umbral de 5.2 elegido por validación cruzada: se elige con 4/5 de las reseñas y
    se mide en el 1/5 restante."""
    clases = np.empty(len(reales), dtype=int)
    for idx_elegir, idx_evaluar in StratifiedKFold(
        n_splits=5, shuffle=True, random_state=42
    ).split(probs, reales):
        mejor = max(
            grilla,
            key=lambda u: (clases_con_umbral(probs[idx_elegir], u) == reales[idx_elegir]).mean(),
        )
        clases[idx_evaluar] = clases_con_umbral(probs[idx_evaluar], mejor)
    return (clases == reales).mean()


reales = test_df["label"].map(etiqueta_a_id).values
comparados = {
    "XLM-RoBERTa": xlm_roberta,
    "XLM-RoBERTa_aumentado": xlm_roberta_aumentado,
    "XLM-RoBERTa_traducido": xlm_roberta_traducido,
}
filas = []
for nombre, resultado in comparados.items():
    aciertos = resultado.probs_test.argmax(axis=1) == reales
    fila = {
        "modelo": nombre,
        "accuracy test_df": aciertos.mean(),
        "accuracy con umbral (CV)": accuracy_umbral_cv(resultado.probs_test, reales),
        "AUC pos/neg": auc_positivo_negativo(resultado.probs_test, test_df),
        "% positivo en eval": (
            resultado.probs_eval.argmax(axis=1) == etiqueta_a_id["positivo"]
        ).mean(),
    }
    # Accuracy en test_df por idioma
    fila.update(
        {
            f"accuracy {idioma}": a
            for idioma, a in pd.Series(aciertos).groupby(test_df["idioma"].values).mean().items()
        }
    )
    filas.append(fila)

display(pd.DataFrame(filas).set_index("modelo").round(4))

In [ ]:
mdeberta_traducido = entrenar_transformer(nombre_modelo="microsoft/mdeberta-v3-base", nombre_corto="mDeBERTa_traducido", train_df=train_df_traducido, test_df=test_df, dataset_eval=dataset_eval)

In [ ]:
errores_mdberta_traducido = evaluar_modelo_validacion(
    mdeberta_traducido.clases_test,
    test_df,
    "mDeBERTa traducido",
    repo_historial=f"{USUARIO_HF}/proyecto-ml-{mdeberta_traducido.nombre_corto}",
)